# 10 — Paper ceiling: ALFRED vintage anchor, net-of-cost economic value, cross-country long–short

Three additions that take the *submitted paper* to its ceiling. Nothing here changes the
identification results; each section produces a pre-registered, reportable outcome either way.

**Required uploads (Colab):**
1. `paper_config_monthly_panel.csv` — the frozen US snapshot from notebook 08 (SHA-256 `67ed3093…d7f5`).
2. `g10_panel_model_objects.csv` — the model-ready panel from notebook 09 v7 (SHA-256 `1628c4ea…bbf6`).
3. **ALFRED vintage file for CPIAUCSL** (any filename): go to
   `https://alfred.stlouisfed.org/series?seid=CPIAUCSL` → *Download Data* → Vintage Dates:
   **Select All** → format **CSV/TXT** → download and upload here. The parser recognises
   `CPIAUCSL_YYYYMMDD` (or `_YYYY-MM-DD`) columns automatically. TB3MS is market data and is
   treated as unrevised (a TB3MS vintage file can be supplied later; not required).

Offline, the synthetic self-test still runs and hard-asserts every identity; real-data cells skip
gracefully per missing input (`US_OK / ALFRED_OK / PANEL_OK`).

### Reading rules fixed before the results

**A. Vintage anchor.** Report canonical vs release-lag(§7) vs full-vintage *as-of* vs *diagonal* on
three rows: basket OOS/CW, 10Y OOS/CW, nested $[1,y_{10}]\to[1,y_{10},TP_{10}]$ CW. Verdict rule:
if the basket $|\Delta R^2_{OOS}|\le 0.010$ **and** the nested CW stays $\ge 1.645$, replace the §7
release-lag sentence with the measured full-vintage row and drop the US pseudo-OOS asterisk;
otherwise report the degradation as a quantified limitation. Both outcomes go in the paper.

**B. Net-of-cost timing.** The reportable line is net-of-cost at **2bp** round-trip per unit
duration-notional (10Y futures territory). Publishable sentence iff net Sharpe(2bp) ≥ static
Sharpe **and** ΔCE(γ=5) > 0; breakeven cost and a skip-month row are reported regardless. This
stays *supporting economic evidence* in §7 — never the publication claim.

**C. Cross-country long–short.** Duration-balanced top-3/bottom-3 on the past-only standardized
signal; legs are local-currency excess returns, hence FX-hedged by construction. Report gross mean,
NW(12) *t*, the median Sharpe over the 12 non-overlapping calendar tracks, net at 1/2/5bp and
breakeven. Labelled **pseudo-out-of-sample** (revised MEI inputs). One table in §7, positioned as
the tradable translation of the TWFE slope.

In [8]:
# CONFIGURATION AND INPUT DISCOVERY
# Colab note: "/content" is the working directory of the temporary Colab VM.
# Files dropped in the left-hand file panel land there, and it is wiped when
# the session ends. If a required input is missing, this cell opens an upload
# dialog so the file can be picked from the local machine; outside Colab it
# prints where to put the file instead.
import os, glob, shutil
from pathlib import Path

CANDIDATE_DIRS = [".", "/content", "macroanchor_data_snapshot",
                  "/content/macroanchor_data_snapshot"]
# Also look one level down, so an unzipped bundle (e.g. the canonical stack or
# the notebook-09 output folder) is found without moving files by hand.
SEARCH_GLOBS = ["*/{name}", "*/*/{name}", "*/*/*/{name}"]

REQUIRED = {
    "paper_config_monthly_panel.csv":
        "US monthly snapshot - needed by every section",
    "g10_panel_model_objects.csv":
        "notebook-09 model panel - needed by Section C only",
}
ALFRED_HINT = ("ALFRED CPI vintage file (any name; recognised by its "
               "CPIAUCSL_<vintage> columns) - needed by Section A only")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except Exception:
    _colab_files = None
    IN_COLAB = False


def _find(name):
    """Locate `name` in the candidate directories, then one to three levels
    below them. Returns the path as a string, or None."""
    for d in CANDIDATE_DIRS:
        p = Path(d) / name
        if p.exists():
            return str(p)
    for d in CANDIDATE_DIRS:
        for pattern in SEARCH_GLOBS:
            hits = sorted(glob.glob(str(Path(d) / pattern.format(name=name))))
            if hits:
                return hits[0]
    return None


def _find_alfred():
    """Find a CSV/TXT whose header carries CPIAUCSL_<vintage> columns."""
    import pandas as pd
    seen = set()
    for d in CANDIDATE_DIRS:
        for pattern in ("*.csv", "*.txt", "*/*.csv", "*/*.txt",
                        "*/*/*.csv", "*/*/*.txt"):
            for p in glob.glob(str(Path(d) / pattern)):
                rp = os.path.realpath(p)
                if rp in seen:
                    continue
                seen.add(rp)
                try:
                    head = pd.read_csv(p, nrows=1)
                except Exception:
                    continue
                if any(str(c).startswith("CPIAUCSL_") for c in head.columns):
                    return p
    return None


def _stage(path):
    """Copy a file found in a sub-folder up into the working directory, so the
    downstream loaders - which only scan the top level of CANDIDATE_DIRS -
    can see it. Returns the path actually in use."""
    if path is None:
        return None
    parent = os.path.realpath(os.path.dirname(os.path.abspath(path)))
    tops = {os.path.realpath(d) for d in CANDIDATE_DIRS if Path(d).exists()}
    if parent in tops:
        return path
    dest = Path(".") / Path(path).name
    if not dest.exists():
        shutil.copy(path, dest)
        print(f"    staged into the working directory: {dest}")
    return str(dest)


def prompt_upload(label):
    """Open the Colab file picker and move whatever is uploaded into the
    working directory. Returns the list of uploaded file names."""
    if not IN_COLAB:
        print(f"    not running on Colab - copy the file next to this "
              f"notebook, or into one of {CANDIDATE_DIRS}, then re-run "
              f"this cell.")
        return []
    print(f"    upload: {label}")
    got = _colab_files.upload()          # opens the browser file dialog
    for name in got:
        if Path(name).exists() and str(Path(name).parent) != ".":
            shutil.move(name, Path(".") / Path(name).name)
    return list(got)


print("input discovery:")
missing = []
for fname, why in REQUIRED.items():
    path = _stage(_find(fname))
    print(f"  {fname:38s} -> {path or 'NOT FOUND'}")
    if path is None:
        missing.append((fname, why))
alfred_path = _stage(_find_alfred())
print(f"  {'ALFRED vintage file':38s} -> {alfred_path or 'NOT FOUND'}")

if missing or alfred_path is None:
    print("\nmissing inputs (each section skips cleanly if its input is "
          "absent, so uploading only what you need is fine):")
    for fname, why in missing:
        print(f"  - {fname}: {why}")
    if alfred_path is None:
        print(f"  - {ALFRED_HINT}")
    if IN_COLAB:
        print("\nopening the upload dialog - press Cancel to skip a file.")
        for fname, why in missing:
            prompt_upload(f"{fname} ({why})")
        if alfred_path is None:
            prompt_upload(ALFRED_HINT)
        print("\nre-checking after upload:")
        for fname in REQUIRED:
            print(f"  {fname:38s} -> "
                  f"{_stage(_find(fname)) or 'still NOT FOUND'}")
        alfred_path = _stage(_find_alfred())
        print(f"  {'ALFRED vintage file':38s} -> "
              f"{alfred_path or 'still NOT FOUND'}")
    else:
        print(f"\nplace the files in one of {CANDIDATE_DIRS} (or in a "
              "sub-folder of those, e.g. an unzipped bundle) and re-run.")
else:
    print("\nall inputs found.")


input discovery:
  paper_config_monthly_panel.csv         -> paper_config_monthly_panel.csv
  g10_panel_model_objects.csv            -> g10_panel_model_objects.csv
  ALFRED vintage file                    -> NOT FOUND

missing inputs (each section skips cleanly if its input is absent, so uploading only what you need is fine):
  - ALFRED CPI vintage file (any name; recognised by its CPIAUCSL_<vintage> columns) - needed by Section A only

opening the upload dialog - press Cancel to skip a file.
    upload: ALFRED CPI vintage file (any name; recognised by its CPIAUCSL_<vintage> columns) - needed by Section A only


Saving CPIAUCSL_all_vintages.csv to CPIAUCSL_all_vintages.csv

re-checking after upload:
  paper_config_monthly_panel.csv         -> paper_config_monthly_panel.csv
  g10_panel_model_objects.csv            -> g10_panel_model_objects.csv
  ALFRED vintage file                    -> CPIAUCSL_all_vintages.csv


In [9]:
# ============================================================================
# LIBRARY — canonical notebook-08 stack (verbatim) + EXT-10 engines
# ============================================================================
# ============================================================================
#  LIBRARY CELL - everything inlined, no local imports needed. Collapse me.
#  Hard requirements: numpy, pandas. Optional: scipy (only for the exact CIR-ML
#  refit, NOT needed by any robustness check), pandas_datareader (FRED fallback
#  to plain CSV works without it).
# ============================================================================
import warnings as _w
_w.filterwarnings('ignore', message='Mean of empty slice')
_w.filterwarnings('ignore', message='All-NaN')
import warnings, io, urllib.request, hashlib, json, shutil, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Constants (identical to the notebooks)
# ----------------------------------------------------------------------------
START = "1961-01-01"
PI_HL = 36            # trend-inflation EWMA half-life (months)
RSTAR_HL = 120        # r* proxy EWMA half-life (months)
MINW = 120            # expanding-window burn-in (months)
H = 12                # forecast horizon (months)
MATS_L = [1, 2, 3, 5, 7, 10]   # paper yield-curve grid (years)
TENORS = [2, 3, 5, 7, 10]      # paper excess-return tenors

# Canonical paper configuration: local frozen inputs, fixed ex ante.
# The CMT snapshot is the canonical July-2026 vintage confirmed for the paper.
PAPER_CURVE_MODE = "gs_cmt_snapshot"
PAPER_RSTAR_MODE = "proxy"
PAPER_EXTERNAL_TP_ID = "THREEFYTP10"
PAPER_ANCHOR_INIT = "post_ewma_bfill"
ALLOW_ECONOMIC_FALLBACKS = False
PAPER_END = pd.Timestamp("2026-05-31")

# Required local inputs. The notebook searches the working directory and /content
# recursively, and refuses files whose byte-level SHA-256 differs.
PAPER_SNAPSHOT_FILENAME = "paper_config_monthly_panel.csv"
PAPER_SNAPSHOT_SHA256 = "67ed3093d17cf5ed849a8cfa3fa95ac5fb2c1f47d58a69110f96a366a1fbd7f5"
PAPER_SNAPSHOT_PANDAS_SHA256 = "c1e32588eb23f57acec28d315973c633852046f68ef6e785cc71657d46989a32"
GSW_LOCAL_FILENAME = "feds200628.csv"
GSW_LOCAL_SHA256 = "c3541ab54e054c0bcc3c17bbe038a3129da2c26486ef3c255e2378d7d63aa9c4"

# Reproducibility and heavier final checks.
WRITE_DATA_SNAPSHOT = True
SNAPSHOT_DIR = Path("macroanchor_data_snapshot")
NOTEBOOK_FILENAME = "08_macroanchor-bond-risk-premia.ipynb"
RUN_ZERO_COUPON_ROBUSTNESS = True
RUN_CONDITIONAL_BOOTSTRAP = True
BOOTSTRAP_B = 9999
BOOTSTRAP_BATCH_SIZE = 250
BOOTSTRAP_BLOCK_LENGTHS = (12, 24)
BOOTSTRAP_SEED = 20260711


# ============================================================================
# 1. DATA LAYER  (from 01/02/03 cells [2]-[3])
# ============================================================================
def fred_series(ids, start=START, end=None):
    """FRED via pandas_datareader, with direct fredgraph CSV fallback."""
    end = end or str(pd.Timestamp.today().date())
    ids = ids if isinstance(ids, list) else [ids]
    try:
        import pandas_datareader.data as web
        df = web.DataReader(ids, "fred", start, end)
        if df is not None and len(df.dropna(how="all")):
            return df
    except Exception as e:
        warnings.warn(f"pandas_datareader failed ({e}); using fredgraph CSV")
    out = {}
    for sid in ids:
        try:
            u = ("https://fred.stlouisfed.org/graph/fredgraph.csv"
                 f"?id={sid}&cosd={pd.Timestamp(start).date()}&coed={pd.Timestamp(end).date()}")
            d = pd.read_csv(u); d.columns = ["date", "val"]
            d["date"] = pd.to_datetime(d["date"])
            s = pd.to_numeric(d.set_index("date")["val"], errors="coerce")
            out[sid] = s.loc[(s.index >= pd.Timestamp(start)) & (s.index <= pd.Timestamp(end))]
        except Exception as e:
            warnings.warn(f"FRED CSV failed for {sid}: {e}")
    return pd.DataFrame(out) if out else None

def _sha256_file(path, chunk_size=1024 * 1024):
    """Byte-level SHA-256 for an immutable input or generated file."""
    path = Path(path); h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def _resolve_local_file(filename, expected_sha256=None):
    """
    Locate a required local file without network fallback. Searches the current
    directory and /content (including extracted subdirectories).
    """
    filename = str(filename)
    roots = [Path.cwd()]
    content = Path("/content")
    if content.exists() and content.resolve() != Path.cwd().resolve():
        roots.append(content)
    candidates = []
    for root in roots:
        direct = root / filename
        if direct.exists():
            candidates.append(direct)
        try:
            candidates.extend(root.rglob(filename))
        except Exception:
            pass
    unique = []
    seen = set()
    for p in candidates:
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key not in seen and p.is_file():
            seen.add(key); unique.append(p)
    if not unique:
        raise FileNotFoundError(
            f"Required local file '{filename}' not found. Upload it to the Colab "
            "session or place it beside the notebook."
        )
    # Prefer the shortest path (normally /content/<filename>).
    path = sorted(unique, key=lambda p: (len(p.parts), str(p)))[0]
    digest = _sha256_file(path)
    if expected_sha256 and digest.lower() != expected_sha256.lower():
        raise RuntimeError(
            f"SHA-256 mismatch for {path}. Expected {expected_sha256}, got {digest}. "
            "Do not resave the file in Excel; use the frozen original."
        )
    return path, digest


def load_paper_snapshot():
    """Load and validate the frozen canonical CMT panel."""
    path, digest = _resolve_local_file(PAPER_SNAPSHOT_FILENAME, PAPER_SNAPSHOT_SHA256)
    df = pd.read_csv(path)
    dcol = next((c for c in df.columns if str(c).strip().lower() == "date"), df.columns[0])
    df[dcol] = pd.to_datetime(df[dcol], errors="raise")
    df = df.set_index(dcol).sort_index()
    df.index.name = "date"
    df = df.loc[df.index <= PAPER_END]
    req = [f"y{m}" for m in MATS_L] + ["TB3MS", "CPIAUCSL", PAPER_EXTERNAL_TP_ID]
    missing = [c for c in req if c not in df.columns]
    if missing:
        raise RuntimeError(f"Canonical snapshot columns missing: {missing}")
    if len(df) != 599 or df.index.min() != pd.Timestamp("1976-06-30") or df.index.max() != PAPER_END:
        raise RuntimeError(
            "Canonical snapshot range mismatch: expected 599 rows, "
            "1976-06-30 to 2026-05-31."
        )
    logical_hash = _hash_frame(df)
    if logical_hash != PAPER_SNAPSHOT_PANDAS_SHA256:
        warnings.warn(
            "Parsed DataFrame hash differs from the archived pandas hash. "
            "The byte-level file hash is valid; this can reflect pandas parsing/version details."
        )
    return df, path, digest, logical_hash


def fetch_gsw(maturities=None):
    """
    Load the frozen local GSW/FEDS zero-coupon file, validate its byte hash,
    convert daily continuously compounded zero yields to month-end decimal units,
    and cut the sample at the canonical paper end date.
    """
    maturities = sorted(set(MATS_L if maturities is None else maturities))
    path, digest = _resolve_local_file(GSW_LOCAL_FILENAME, GSW_LOCAL_SHA256)
    raw = path.read_text(encoding="utf-8", errors="replace")
    lines = raw.splitlines()
    try:
        hdr = next(i for i, line in enumerate(lines) if line.lower().startswith("date,"))
    except StopIteration as exc:
        raise RuntimeError("Could not locate the GSW Date header") from exc
    df = pd.read_csv(io.StringIO("\n".join(lines[hdr:])), na_values=["NA", -999.99, "-999.99"])
    df.columns = [str(c).strip() for c in df.columns]
    dc = df.columns[0]
    df[dc] = pd.to_datetime(df[dc], errors="coerce")
    df = df.dropna(subset=[dc]).set_index(dc).sort_index()
    need = [f"SVENY{m:02d}" for m in maturities]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise RuntimeError(f"GSW columns missing: {missing}")
    out = df[need].apply(pd.to_numeric, errors="coerce").resample("ME").last() / 100.0
    out = out.loc[out.index <= PAPER_END]
    out.columns = maturities
    out.attrs["source_path"] = str(path)
    out.attrs["sha256_file"] = digest
    return out

def fetch_tp(series_id=PAPER_EXTERNAL_TP_ID):
    """Fetch one pre-specified external 10Y term-premium series; never substitute silently."""
    if series_id is None:
        return None, None
    d = fred_series([series_id])
    if d is None or series_id not in d or d[series_id].notna().sum() <= 50:
        raise RuntimeError(f"Required external TP series unavailable: {series_id}")
    s = d[series_id].resample("ME").last() / 100.0
    s.name = series_id
    return s, series_id

def fetch_hlw():
    """Best-effort HLW r* (NY Fed CSV). -> (Series decimal, src) or (None, None)."""
    urls = ["https://www.newyorkfed.org/medialibrary/media/research/policy/rstar/"
            "Holston_Laubach_Williams_current_estimates.csv"]
    for u in urls:
        try:
            df = pd.read_csv(u)
            dcol = [c for c in df.columns if "date" in c.lower()][0]
            rcands = [c for c in df.columns if "rstar" in c.lower()]
            rcol = ([c for c in rcands if "us" in c.lower()] or rcands)[0]
            s = pd.Series(pd.to_numeric(df[rcol], errors="coerce").values,
                          index=pd.to_datetime(df[dcol])).dropna()
            s = (s / 100.0 if s.median() > 1 else s).resample("ME").last().interpolate()
            if len(s) > 40:
                return s, "HLW (NY Fed)"
        except Exception:
            pass
    return None, None


def load_us_data(curve_mode=PAPER_CURVE_MODE, start=START,
                 external_tp_id=PAPER_EXTERNAL_TP_ID,
                 rstar_mode=PAPER_RSTAR_MODE,
                 allow_fallbacks=ALLOW_ECONOMIC_FALLBACKS):
    """
    Load a fixed local economic specification.

    `gs_cmt_snapshot` reads the frozen canonical CMT panel. `gsw_zero` replaces
    only the yield curve with the frozen local GSW zero-coupon curve while using
    the same TB3MS, CPI and Kim-Wright vintage. No economic fallback is permitted.
    """
    if allow_fallbacks:
        raise ValueError("Economic fallbacks are disabled for the publication notebook")
    curve_mode = str(curve_mode).lower(); rstar_mode = str(rstar_mode).lower()
    snap, snap_path, snap_digest, snap_logical_hash = load_paper_snapshot()

    short_s = pd.to_numeric(snap["TB3MS"], errors="coerce")
    cpi_s = pd.to_numeric(snap["CPIAUCSL"], errors="coerce")
    tp_series = tp_name = None
    if external_tp_id is not None:
        if external_tp_id not in snap.columns:
            raise RuntimeError(f"External TP {external_tp_id} absent from canonical snapshot")
        tp_series = pd.to_numeric(snap[external_tp_id], errors="coerce")
        tp_series.name = external_tp_id; tp_name = external_tp_id

    cp_curve = None; gsw_path = gsw_digest = None
    if curve_mode in ("gs_cmt_snapshot", "gs_cmt"):
        curve = snap[[f"y{m}" for m in MATS_L]].copy()
        curve.columns = MATS_L
        curve_src = "Frozen FRED GS constant-maturity snapshot (par-equivalent)"
        curve_kind = "cmt_par_equivalent"
    elif curve_mode == "gsw_zero":
        full = fetch_gsw(sorted(set(MATS_L + [4])))
        gsw_path = full.attrs.get("source_path")
        gsw_digest = full.attrs.get("sha256_file")
        curve = full[MATS_L].copy()
        cp_curve = full[[1, 2, 3, 4, 5]].copy()
        curve_src = "Frozen Fed GSW zero-coupon (feds200628)"
        curve_kind = "zero_coupon"
    else:
        raise ValueError("curve_mode must be 'gs_cmt_snapshot' or 'gsw_zero'")

    if rstar_mode == "proxy":
        hlw = hlw_src = None
    elif rstar_mode == "hlw":
        # Retained for development only; not used by the canonical notebook.
        hlw, hlw_src = fetch_hlw()
        if hlw is None:
            raise RuntimeError("HLW requested but unavailable")
    else:
        raise ValueError("rstar_mode must be 'proxy' or 'hlw'")

    idx = snap.index.intersection(curve.dropna().index)
    idx = idx[(idx >= pd.Timestamp("1976-06-30")) & (idx <= PAPER_END)]
    dates = pd.DatetimeIndex(idx)
    Ydf = curve.reindex(dates).interpolate(limit_area="inside")
    short_al = short_s.reindex(dates).interpolate(limit_area="inside")
    cpi_al = cpi_s.reindex(dates).interpolate(limit_area="inside")
    if Ydf.isna().any().any() or short_al.isna().any() or cpi_al.isna().any():
        raise RuntimeError("Unresolved missing values in the fixed panel")
    if tp_series is not None:
        tp_series = tp_series.reindex(dates)
    if hlw is not None:
        hlw = hlw.reindex(dates)
    if cp_curve is not None:
        cp_curve = cp_curve.reindex(dates).interpolate(limit_area="inside")
        if cp_curve.isna().any().any():
            raise RuntimeError("Unresolved missing values in annual 1-5Y GSW curve")

    raw_panel = Ydf.copy(); raw_panel.columns = [f"y{m}" for m in MATS_L]
    raw_panel["TB3MS"] = short_al; raw_panel["CPIAUCSL"] = cpi_al
    if tp_series is not None:
        raw_panel[tp_name] = tp_series

    return dict(
        dates=dates, Y=Ydf.to_numpy(), short=short_al.to_numpy(),
        cpi_idx=cpi_al.to_numpy(), tp_series=tp_series, tp_name=tp_name,
        hlw=hlw, hlw_src=hlw_src, curve_src=curve_src,
        curve_kind=curve_kind, curve_mode=curve_mode,
        cp_curve=None if cp_curve is None else cp_curve.to_numpy(),
        cp_taus=None if cp_curve is None else np.array([1, 2, 3, 4, 5], float),
        raw_panel=raw_panel,
        canonical_input_file=str(snap_path),
        canonical_input_sha256_file=snap_digest,
        canonical_input_sha256_pandas=snap_logical_hash,
        gsw_input_file=gsw_path,
        gsw_input_sha256_file=gsw_digest,
    )

# ============================================================================
# 2. THE MACRO ANCHOR  theta_t = r*_t + pi_e,t   (from cell [4])
# ============================================================================
def build_anchor(dates, short, cpi_idx, hlw=None, pi_hl=PI_HL,
                 rstar_hl=RSTAR_HL, initialization=PAPER_ANCHOR_INIT):
    """
    theta_t = r*_t + pi_e,t.

    `post_ewma_bfill` is the new canonical specification: both EWMAs are computed
    from available history first, and only their final initial gaps are backfilled.
    `strict` leaves initial values missing. `draft_pre_rstar_bfill` reproduces the
    old draft ordering in which trend inflation was backfilled before entering r*.
    """
    cpi_yoy = pd.Series(cpi_idx, index=dates).pct_change(12)
    pi_raw = cpi_yoy.ewm(halflife=pi_hl).mean()
    short_s = pd.Series(short, index=dates)

    if hlw is not None:
        pi_trend = pi_raw
        rstar = pd.Series(np.asarray(hlw), index=dates).reindex(dates).interpolate(limit_area="inside").ffill()
        rstar_src = "HLW (NY Fed)"
        if initialization in ("post_ewma_bfill", "draft_pre_rstar_bfill"):
            pi_trend = pi_trend.bfill(); rstar = rstar.bfill()
        elif initialization != "strict":
            raise ValueError("Unknown anchor initialization")
    else:
        if initialization == "draft_pre_rstar_bfill":
            pi_trend = pi_raw.bfill()
            rstar = (short_s - pi_trend).ewm(halflife=rstar_hl).mean().bfill()
        else:
            pi_trend = pi_raw
            rstar = (short_s - pi_trend).ewm(halflife=rstar_hl).mean()
            if initialization == "post_ewma_bfill":
                pi_trend = pi_trend.bfill(); rstar = rstar.bfill()
            elif initialization != "strict":
                raise ValueError(
                    "initialization must be 'post_ewma_bfill', 'strict', "
                    "or 'draft_pre_rstar_bfill'"
                )
        rstar_src = f"proxy: EWMA real short (hl={rstar_hl}m)"
    anchor_sr = (rstar + pi_trend).to_numpy()
    return anchor_sr, rstar.to_numpy(), pi_trend.to_numpy(), rstar_src

# ============================================================================
# 3. PHYSICAL-MEASURE ESTIMATION  (Section A, cell [5])
# ============================================================================
DTm = 1 / 12


def fit_ou(rw):
    """OU/Vasicek by exact AR(1) discretisation, with finite-pair masking."""
    rw = np.asarray(rw, float)
    m = np.isfinite(rw[:-1]) & np.isfinite(rw[1:])
    x = rw[:-1][m]; yv = rw[1:][m]
    if len(x) < 24: return np.nan, np.nan, np.nan
    X = np.column_stack([np.ones(len(x)), x]); c, b = np.linalg.lstsq(X, yv, rcond=None)[0]
    b = float(np.clip(b, 1e-6, 0.999999)); k = -12 * np.log(b); th = float(c / (1 - b))
    e = yv - (c + b * x); s2 = float(np.var(e)) * 2 * k / max(1 - b * b, 1e-10)
    return k, th, np.sqrt(max(s2, 1e-10))

def fit_cir(rw):
    """CIR weighted least squares (QML moments), with finite-pair masking."""
    rw = np.asarray(rw, float)
    m = np.isfinite(rw[:-1]) & np.isfinite(rw[1:])
    x0 = rw[:-1][m]; x1 = rw[1:][m]
    if len(x0) < 24: return np.nan, np.nan, np.nan
    x = np.maximum(x0, 1e-4); d = x1 - x0; w = 1 / np.sqrt(x)
    X = np.column_stack([np.ones(len(x)) * w, x * w]); yv = d * w
    b0, b1 = np.linalg.lstsq(X, yv, rcond=None)[0]
    k = float(np.clip(-b1 * 12, 1e-4, 20)); th = float(np.clip(b0 * 12 / k, 1e-4, 0.5))
    e = d - (b0 + b1 * x); s2 = float(np.mean(e * e / (x * DTm)))
    return k, th, np.sqrt(max(s2, 1e-10))

def cir_ml_refit(r, k0, th0, s0):
    """Exact non-central chi-square ML refinement; returns input if unavailable."""
    if not np.all(np.isfinite([k0, th0, s0])):
        return k0, th0, s0
    try:
        from scipy.stats import ncx2
        from scipy.optimize import minimize
        r = np.asarray(r, float); m = np.isfinite(r[:-1]) & np.isfinite(r[1:])
        r0, r1 = r[:-1][m], r[1:][m]
        def nll(p):
            k, th, s = p
            if k <= 1e-4 or th <= 1e-4 or s <= 1e-4: return 1e9
            c = 2 * k / (s * s * (1 - np.exp(-k * DTm))); df = 4 * k * th / (s * s)
            nc = 2 * c * np.maximum(r0, 1e-6) * np.exp(-k * DTm)
            ll = ncx2.logpdf(2 * c * np.maximum(r1, 1e-6), df, nc) + np.log(2 * c)
            return -np.nansum(ll)
        res = minimize(nll, [k0, th0, s0], method="Nelder-Mead",
                       options={"maxiter": 400, "xatol": 1e-5, "fatol": 1e-3})
        if res.success or res.fun < nll([k0, th0, s0]):
            return tuple(float(v) for v in res.x)
    except Exception:
        pass
    return k0, th0, s0

def fit_hw(rw, anc):
    """HW gap regression with finite-pair masking. Returns (kappa, sigma, t-gap)."""
    rw = np.asarray(rw, float); anc = np.asarray(anc, float)
    m = np.isfinite(rw[:-1]) & np.isfinite(rw[1:]) & np.isfinite(anc[:-1])
    g = anc[:-1][m] - rw[:-1][m]; d = rw[1:][m] - rw[:-1][m]
    if len(g) < 24 or np.sum((g-g.mean())**2) <= 0:
        return np.nan, np.nan, np.nan
    X = np.column_stack([np.ones(len(g)), g]); a0, sl = np.linalg.lstsq(X, d, rcond=None)[0]
    e = d - (a0 + sl * g)
    seb = np.sqrt(np.sum(e * e) / max(len(g) - 2, 1) / np.sum((g - g.mean()) ** 2))
    k = float(np.clip(sl * 12, 1e-4, 20)); s = float(np.std(e)) * np.sqrt(12)
    return k, s, float(sl / seb) if seb > 0 else np.nan

def expanding_params(r, anchor_sr, minw=MINW, cir_ml=True):
    """Expanding-window OU/CIR/HW parameters (no look-ahead). Returns a dict of arrays."""
    Tn = len(r)
    out = {k: np.full(Tn, np.nan) for k in
           ["kOU", "thOU", "sOU", "kC", "thC", "sC", "kHW", "sHW"]}
    for t in range(minw, Tn):
        a, b, c_ = fit_ou(r[:t + 1]); out["kOU"][t], out["thOU"][t], out["sOU"][t] = a, b, c_
        a, b, c_ = fit_cir(r[:t + 1])
        if cir_ml and np.all(np.isfinite([a, b, c_])):
            a, b, c_ = cir_ml_refit(r[:t + 1], a, b, c_)
        out["kC"][t], out["thC"][t], out["sC"][t] = a, b, c_
        a, c_, _ = fit_hw(r[:t + 1], anchor_sr[:t + 1]); out["kHW"][t], out["sHW"][t] = a, c_
    return out


# ============================================================================
# 4. PURE-EXPECTATIONS CURVES -> MODEL TERM PREMIA  (Section B, cell [6])
# ============================================================================
def vas_B(tau, k):
    k = max(k, 1e-4); return (1 - np.exp(-k * tau)) / k


def vas_yield(tau, rr, k, th, s):
    k = max(k, 1e-4); B = vas_B(tau, k)
    lnA = (th - s * s / (2 * k * k)) * (B - tau) - s * s * B * B / (4 * k)
    return (-lnA + B * rr) / tau


def cir_yield(tau, rr, k, th, s):
    k = max(k, 1e-4); s = max(s, 1e-4); g = np.sqrt(k * k + 2 * s * s)
    eg = np.exp(g * tau); den = (g + k) * (eg - 1) + 2 * g
    B = 2 * (eg - 1) / den; lnA = (2 * k * th / (s * s)) * np.log(2 * g * np.exp((k + g) * tau / 2) / den)
    return (-lnA + B * rr) / tau


def model_term_premia(Y, r, anchor_sr, params, taus=None, minw=MINW):
    """Per-tenor model TP = y_obs - y_PE for HW / OU / CIR. Returns (TPHW, TPOU, TPCIR)."""
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    Tn, nta = len(r), len(taus)
    TPHW = np.full((Tn, nta), np.nan); TPOU = np.full((Tn, nta), np.nan); TPCIR = np.full((Tn, nta), np.nan)
    for t in range(minw, Tn):
        for j, tau in enumerate(taus):
            if np.all(np.isfinite([Y[t, j], r[t], params["kOU"][t], params["thOU"][t], params["sOU"][t]])):
                TPOU[t, j] = Y[t, j] - vas_yield(tau, r[t], params["kOU"][t], params["thOU"][t], params["sOU"][t])
            if np.all(np.isfinite([Y[t, j], r[t], params["kHW"][t], anchor_sr[t], params["sHW"][t]])):
                TPHW[t, j] = Y[t, j] - vas_yield(tau, r[t], params["kHW"][t], anchor_sr[t], params["sHW"][t])
            if np.all(np.isfinite([Y[t, j], r[t], params["kC"][t], params["thC"][t], params["sC"][t]])):
                TPCIR[t, j] = Y[t, j] - cir_yield(tau, r[t], params["kC"][t], params["thC"][t], params["sC"][t])
    return TPHW, TPOU, TPCIR


# ============================================================================
# 5. RETURNS AND PREDICTORS  (Section C, cell [8])
# ============================================================================
def y_at(Y, taus, m):
    return np.array([np.interp(m, taus, Y[t]) for t in range(len(Y))])


def excess_returns_fb(Y, taus=None, tenors=None, H=H):
    """
    Zero-coupon Fama-Bliss holding-period returns when Y is a zero-coupon curve.
    With GS constant-maturity/par-equivalent inputs this is a CMT-based FB-style
    approximation and must be labelled as such.
    """
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    tenors = TENORS if tenors is None else list(tenors)
    Tn = len(Y); y1 = y_at(Y, taus, 1); RX = {}
    for n in tenors:
        yn = y_at(Y, taus, n); ynm1 = y_at(Y, taus, n - 1); rx = np.full(Tn, np.nan)
        for t in range(Tn - H):
            if np.all(np.isfinite([yn[t], ynm1[t + H], y1[t]])):
                rx[t] = n * yn[t] - (n - 1) * ynm1[t + H] - y1[t]
        RX[n] = rx
    rx_bar = np.nanmean(np.column_stack([RX[n] for n in tenors]), axis=1)
    return RX, rx_bar

def monthly_returns(Y, r, taus=None):
    """Per-tenor 1m carry+roll+duration+convexity returns; basket excess-of-cash."""
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    Tn, nta = len(Y), len(taus); Dm = taus; Cm = taus ** 2; hh = 1 / 12
    dYm = np.vstack([np.zeros((1, nta)), np.diff(Y, axis=0)])
    rollm = np.zeros_like(Y)
    for t in range(Tn):
        rollm[t] = Dm * (Y[t] - np.interp(taus - hh, taus, Y[t]))
    RET_pt = Y / 12 + rollm - Dm[None, :] * dYm + 0.5 * Cm[None, :] * dYm ** 2
    basket1m = RET_pt[:, 1:].mean(1) - r / 12
    return RET_pt, basket1m


def fama_bliss_fwd(Y, taus=None):
    """Average forward spread (the notebook's fb_bar)."""
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    y1 = Y[:, 0]; fs = []
    for j in range(1, len(taus)):
        f = (taus[j] * Y[:, j] - taus[j - 1] * Y[:, j - 1]) / (taus[j] - taus[j - 1])
        fs.append(f - y1)
    return np.column_stack(fs).mean(1)


def forward_vector(Y, taus=None):
    """
    Forward-rate regressor matrix. On the irregular paper grid this produces a
    CP-style factor (some intervals are multi-year averages). With annual 1..5Y
    zero-coupon inputs it is the canonical one-year-forward construction.
    """
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    cols = [y_at(Y, taus, 1)]
    for j in range(1, len(taus)):
        f = (taus[j] * Y[:, j] - taus[j - 1] * Y[:, j - 1]) / (taus[j] - taus[j - 1])
        cols.append(f)
    return np.column_stack(cols)

def legacy_gap(Y, anchor_sr, tp_add=0.005):
    """The failed raw macro-anchor gap: y10 - (anchor + fixed TP block)."""
    return Y[:, -1] - (anchor_sr + tp_add)


# ============================================================================
# 6. INFERENCE  (Section C + Clark-West cells [8],[14],[16]) -- unified
# ============================================================================
def ols_nw(yv, Xc, L):
    """OLS with Newey-West SE. Xc is a list of regressor columns. Returns (b, se, R2)."""
    X = np.column_stack([np.ones(len(yv))] + [c for c in Xc]); b = np.linalg.lstsq(X, yv, rcond=None)[0]
    e = yv - X @ b; XtX = np.linalg.inv(X.T @ X); Xe = X * e[:, None]; S = Xe.T @ Xe
    for l in range(1, L + 1):
        w = 1 - l / (L + 1); G = Xe[l:].T @ Xe[:-l]; S = S + w * (G + G.T)
    V = XtX @ S @ XtX
    return b, np.sqrt(np.diag(V)), 1 - (e @ e) / (((yv - yv.mean()) ** 2).sum())


def is_r2(P, yv):
    f = (np.isfinite(yv) & np.isfinite(P).all(1)) if P.ndim > 1 else (np.isfinite(yv) & np.isfinite(P))
    Pm = P[f].reshape(-1, 1) if P.ndim == 1 else P[f]; X = np.column_stack([np.ones(f.sum()), Pm])
    b = np.linalg.lstsq(X, yv[f], rcond=None)[0]; e = yv[f] - X @ b
    return 1 - (e @ e) / (((yv[f] - yv[f].mean()) ** 2).sum())


def first_valid_start(P, yv, H=H, minobs=MINW):
    """First origin t where an expanding fit is possible (used to define the common window)."""
    P = np.asarray(P, float); P2 = P.reshape(-1, 1) if P.ndim == 1 else P; yv = np.asarray(yv, float)
    for t in range(minobs + H, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P2[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P2[s_]).all(1)
        if m.sum() >= minobs:
            return t
    return None


def oos_r2(P, yv, H=H, minobs=MINW, start=None):
    """
    Campbell-Thompson OOS R2 vs the expanding mean. Unifies oos_r2 / oos_r2_w:
    start=None -> native (first valid origin); start=t0 -> forced common window.
    Returns (R2, n_forecasts).
    """
    P = np.asarray(P, float); P = P.reshape(-1, 1) if P.ndim == 1 else P; yv = np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    num = den = 0.0; nf = 0
    for t in range(t_start, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P[s_]).all(1)
        if m.sum() < minobs:
            continue
        Xtr = np.column_stack([np.ones(m.sum()), P[s_][m]]); btr = np.linalg.lstsq(Xtr, yv[s_][m], rcond=None)[0]
        num += (yv[t] - (btr[0] + P[t] @ btr[1:])) ** 2; den += (yv[t] - yv[s_][m].mean()) ** 2; nf += 1
    return (1 - num / den if den > 0 else np.nan), nf


def clark_west(P, yv, H=H, minobs=MINW, nwlag=12, start=None):
    """
    Clark-West (2007) adjusted-MSPE test vs the expanding mean (nested).
    One-sided; NW(nwlag). start unifies clark_west / cw_w. Returns (adjMSPE, t, p, n).
    """
    from statistics import NormalDist
    P = np.asarray(P, float); P = P.reshape(-1, 1) if P.ndim == 1 else P; yv = np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    fts = []
    for t in range(t_start, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P[s_]).all(1)
        if m.sum() < minobs:
            continue
        yb = yv[s_][m].mean()
        Xtr = np.column_stack([np.ones(m.sum()), P[s_][m]]); btr = np.linalg.lstsq(Xtr, yv[s_][m], rcond=None)[0]
        ym = btr[0] + P[t] @ btr[1:]; eb = yv[t] - yb; em = yv[t] - ym
        fts.append(eb * eb - (em * em - (yb - ym) ** 2))
    f = np.array(fts, float); n = len(f)
    if n < 24:
        return np.nan, np.nan, np.nan, n
    u = f - f.mean(); S = np.sum(u * u) / n
    for l in range(1, nwlag + 1):
        w = 1 - l / (nwlag + 1); S += 2 * w * np.sum(u[l:] * u[:-l]) / n
    se = np.sqrt(S / n); tcw = f.mean() / se if se > 0 else np.nan
    p = (1 - NormalDist().cdf(tcw)) if np.isfinite(tcw) else np.nan
    return f.mean(), tcw, p, n


# ============================================================================
# 7. NEW ROBUSTNESS HELPERS  (notebook 08)
# ============================================================================
def orthogonalize_expanding(P, Z, minobs=MINW):
    """
    Strictly past-only expanding residualisation. At origin t, coefficients are
    fitted on s < t; the current observation is projected using those loadings.
    """
    P = np.asarray(P, float); Z = np.asarray(Z, float)
    Z = Z.reshape(-1, 1) if Z.ndim == 1 else Z
    Tn = len(P); out = np.full(Tn, np.nan)
    for t in range(minobs, Tn):
        if not (np.isfinite(P[t]) and np.isfinite(Z[t]).all()): continue
        s_ = np.arange(0, t)
        m = np.isfinite(P[s_]) & np.isfinite(Z[s_]).all(1)
        if m.sum() < minobs: continue
        X = np.column_stack([np.ones(m.sum()), Z[s_][m]])
        b = np.linalg.lstsq(X, P[s_][m], rcond=None)[0]
        out[t] = P[t] - (b[0] + Z[t] @ b[1:])
    return out

def nested_oos_cw(P_small, P_big, yv, H=H, minobs=MINW, nwlag=12, start=None):
    """
    OOS-R2 (big vs small as the benchmark) AND Clark-West for a nested pair of
    predictor SETS. P_small / P_big are (n,) or (n,k) arrays; columns of P_small
    must be a subset of P_big for the CW adjustment to be valid.

    Use to ask "does adding TP to [1, y10] beat [1, y10] out of sample?".
    Returns dict(OOS_R2_big_vs_small, OOS_R2_big_vs_mean, CW_t, CW_p, n).
    """
    from statistics import NormalDist

    def _mat(P):
        P = np.asarray(P, float); return P.reshape(-1, 1) if P.ndim == 1 else P
    Ps, Pb, yv = _mat(P_small), _mat(P_big), np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    num_bs = den_bs = num_bm = den_bm = 0.0; fts = []; nf = 0
    for t in range(t_start, len(yv)):
        if not (np.isfinite(yv[t]) and np.isfinite(Ps[t]).all() and np.isfinite(Pb[t]).all()):
            continue
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(yv[s_]) & np.isfinite(Ps[s_]).all(1) & np.isfinite(Pb[s_]).all(1)
        if m.sum() < minobs:
            continue
        ytr = yv[s_][m]
        Xs = np.column_stack([np.ones(m.sum()), Ps[s_][m]]); bs = np.linalg.lstsq(Xs, ytr, rcond=None)[0]
        Xb = np.column_stack([np.ones(m.sum()), Pb[s_][m]]); bb = np.linalg.lstsq(Xb, ytr, rcond=None)[0]
        f_s = bs[0] + Ps[t] @ bs[1:]; f_b = bb[0] + Pb[t] @ bb[1:]; ybar = ytr.mean()
        e_s = yv[t] - f_s; e_b = yv[t] - f_b; e_m = yv[t] - ybar
        num_bs += e_b ** 2; den_bs += e_s ** 2
        num_bm += e_b ** 2; den_bm += e_m ** 2
        # CW for the (small) benchmark vs (big) alternative
        fts.append(e_s * e_s - (e_b * e_b - (f_s - f_b) ** 2)); nf += 1
    f = np.array(fts, float); n = len(f)
    if n < 24:
        return dict(OOS_R2_big_vs_small=np.nan, OOS_R2_big_vs_mean=np.nan, CW_t=np.nan, CW_p=np.nan, n=n)
    u = f - f.mean(); S = np.sum(u * u) / n
    for l in range(1, nwlag + 1):
        w = 1 - l / (nwlag + 1); S += 2 * w * np.sum(u[l:] * u[:-l]) / n
    se = np.sqrt(S / n); tcw = f.mean() / se if se > 0 else np.nan
    p = (1 - NormalDist().cdf(tcw)) if np.isfinite(tcw) else np.nan
    return dict(OOS_R2_big_vs_small=(1 - num_bs / den_bs) if den_bs > 0 else np.nan,
                OOS_R2_big_vs_mean=(1 - num_bm / den_bm) if den_bm > 0 else np.nan,
                CW_t=tcw, CW_p=p, n=n)


def cp_factor_expanding(Y, rx_bar, taus=None, minobs=MINW, H=H):
    """
    Expanding CP-style fitted forward factor. It is canonical CP only when `Y`
    contains annual 1..5Y zero-coupon yields and rx_bar is the 2..5Y average.
    Forecast loadings use only targets realised by the origin (s+H <= t).
    """
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    F = forward_vector(Y, taus); Tn = len(Y); out = np.full(Tn, np.nan)
    rx_bar = np.asarray(rx_bar, float)
    for t in range(minobs, Tn):
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(rx_bar[s_]) & np.isfinite(F[s_]).all(1)
        if m.sum() < minobs or not np.isfinite(F[t]).all(): continue
        X = np.column_stack([np.ones(m.sum()), F[s_][m]])
        g = np.linalg.lstsq(X, rx_bar[s_][m], rcond=None)[0]
        out[t] = g[0] + F[t] @ g[1:]
    return out

# ============================================================================
# 8. CONVENIENCE: run the whole US core in one call
# ============================================================================
def build_us_core(curve_mode=PAPER_CURVE_MODE, rstar_mode=PAPER_RSTAR_MODE,
                  external_tp_id=PAPER_EXTERNAL_TP_ID,
                  anchor_init=PAPER_ANCHOR_INIT,
                  allow_fallbacks=ALLOW_ECONOMIC_FALLBACKS,
                  minw=MINW, cir_ml=False, verbose=True):
    """Fixed-configuration assembly: data -> anchor -> model signal -> targets."""
    d = load_us_data(curve_mode=curve_mode, external_tp_id=external_tp_id,
                     rstar_mode=rstar_mode, allow_fallbacks=allow_fallbacks)
    dates, Y, short, cpi_idx = d["dates"], d["Y"], d["short"], d["cpi_idx"]
    r = np.asarray(short, float)
    anchor_sr, rstar, pi_trend, rstar_src = build_anchor(
        dates, short, cpi_idx, hlw=d["hlw"], initialization=anchor_init)
    params = expanding_params(r, anchor_sr, minw=minw, cir_ml=cir_ml)
    taus = np.array(MATS_L, float)
    TPHW, TPOU, TPCIR = model_term_premia(Y, r, anchor_sr, params, taus=taus, minw=minw)
    RX, rx_bar = excess_returns_fb(Y, taus=taus)
    RET_pt, basket1m = monthly_returns(Y, r, taus=taus)
    tpHW_bar = np.where(np.isfinite(TPHW[:, 1:]).all(1), np.mean(TPHW[:, 1:], axis=1), np.nan)
    tpOU_bar = np.where(np.isfinite(TPOU[:, 1:]).all(1), np.mean(TPOU[:, 1:], axis=1), np.nan)
    slope = Y[:, -1] - Y[:, 0]
    fb_bar = fama_bliss_fwd(Y, taus=taus); gap = legacy_gap(Y, anchor_sr)
    target_label = ("zero-coupon Fama-Bliss" if d["curve_kind"] == "zero_coupon"
                    else "CMT-based Fama-Bliss-style")
    if verbose:
        print(f"Assembled {len(dates)} months: {dates[0].date()} -> {dates[-1].date()}")
        print(f"  curve: {d['curve_src']} | r*: {rstar_src} | TP ref: {d['tp_name'] or 'none'}")
        print(f"  target interpretation: {target_label} | anchor init: {anchor_init}")
    return dict(dates=dates, Y=Y, r=r, taus=taus, anchor_sr=anchor_sr,
                rstar=rstar, pi_trend=pi_trend, params=params,
                TPHW=TPHW, TPOU=TPOU, TPCIR=TPCIR, tpHW_bar=tpHW_bar, tpOU_bar=tpOU_bar,
                RX=RX, rx_bar=rx_bar, RET_pt=RET_pt, basket1m=basket1m,
                slope=slope, fb_bar=fb_bar, gap=gap,
                tp_series=d["tp_series"], tp_name=d["tp_name"],
                target_label=target_label, meta=d)


# ============================================================================
# 10. PUBLICATION HELPERS: matched windows, snapshots and finite-sample checks
# ============================================================================
def _as2d(P):
    P = np.asarray(P, float)
    return P.reshape(-1, 1) if P.ndim == 1 else P


def matched_start(*args, yv, H=H, minobs=MINW):
    starts = [first_valid_start(np.asarray(P, float), yv, H=H, minobs=minobs) for P in args]
    starts = [s for s in starts if s is not None]
    return max(starts) if starts else None


def paired_benchmark_table(tp, benchmarks, yv, dates, tp_name="TP_HW", H=H, minobs=MINW):
    """Each benchmark is compared with TP on its own matched window; both incremental directions."""
    rows = []
    tp = np.asarray(tp, float); yv = np.asarray(yv, float)
    for bname, B in benchmarks:
        B = np.asarray(B, float); st = matched_start(tp, B, yv=yv, H=H, minobs=minobs)
        if st is None: continue
        otp, ntp = oos_r2(tp, yv, start=st); _, ctp, ptp, _ = clark_west(tp, yv, start=st)
        ob, nb = oos_r2(B, yv, start=st); _, cb, pb, _ = clark_west(B, yv, start=st)
        add_tp = nested_oos_cw(B, np.column_stack([B, tp]), yv, start=st)
        add_b = nested_oos_cw(tp, np.column_stack([tp, B]), yv, start=st)
        rows.append(dict(benchmark=bname, start=str(pd.DatetimeIndex(dates)[st].date()),
                         n=min(ntp, nb), TP_OOS=otp, TP_CW=ctp,
                         B_OOS=ob, B_CW=cb,
                         TP_adds_R2=add_tp["OOS_R2_big_vs_small"], TP_adds_CW=add_tp["CW_t"],
                         B_adds_R2=add_b["OOS_R2_big_vs_small"], B_adds_CW=add_b["CW_t"]))
    return pd.DataFrame(rows)


def nested_oos_errors(P_small, P_big, yv, H=H, minobs=MINW, start=None):
    """Per-origin squared errors for a nested pair, for regime attribution."""
    Ps, Pb, yv = _as2d(P_small), _as2d(P_big), np.asarray(yv, float)
    t_start = (minobs + H) if start is None else start
    idx=[]; es2=[]; eb2=[]
    for t in range(t_start, len(yv)):
        if not (np.isfinite(yv[t]) and np.isfinite(Ps[t]).all() and np.isfinite(Pb[t]).all()): continue
        s_ = np.arange(0, t-H+1)
        m = np.isfinite(yv[s_]) & np.isfinite(Ps[s_]).all(1) & np.isfinite(Pb[s_]).all(1)
        if m.sum() < minobs: continue
        ys = yv[s_][m]
        Xs=np.column_stack([np.ones(m.sum()), Ps[s_][m]]); bs=np.linalg.lstsq(Xs,ys,rcond=None)[0]
        Xb=np.column_stack([np.ones(m.sum()), Pb[s_][m]]); bb=np.linalg.lstsq(Xb,ys,rcond=None)[0]
        fs=bs[0]+Ps[t]@bs[1:]; fb=bb[0]+Pb[t]@bb[1:]
        idx.append(t); es2.append((yv[t]-fs)**2); eb2.append((yv[t]-fb)**2)
    return np.asarray(idx,int), np.asarray(es2,float), np.asarray(eb2,float)


def _hash_frame(df):
    x = pd.util.hash_pandas_object(df, index=True).values.tobytes()
    return hashlib.sha256(x).hexdigest()


def write_data_snapshot(core, directory=SNAPSHOT_DIR, label="paper_config"):
    """Write the exact in-memory panel and a provenance-rich SHA-256 manifest."""
    directory = Path(directory); directory.mkdir(parents=True, exist_ok=True)
    df = core["meta"]["raw_panel"].copy()
    csv = directory / f"{label}_monthly_panel.csv"
    df.to_csv(csv, index_label="date")
    manifest = {
        "label": label,
        "rows": int(len(df)),
        "start": str(df.index.min().date()),
        "end": str(df.index.max().date()),
        "sha256_pandas": _hash_frame(df),
        "sha256_file": _sha256_file(csv),
        "csv_filename": csv.name,
        "curve_source": core["meta"]["curve_src"],
        "curve_kind": core["meta"]["curve_kind"],
        "rstar_mode": PAPER_RSTAR_MODE,
        "external_tp": core["tp_name"],
        "anchor_initialization": PAPER_ANCHOR_INIT,
        "canonical_input_file": Path(core["meta"]["canonical_input_file"]).name,
        "canonical_input_sha256_file": core["meta"]["canonical_input_sha256_file"],
        "canonical_input_sha256_pandas": core["meta"]["canonical_input_sha256_pandas"],
        "gsw_input_file": (
            Path(core["meta"]["gsw_input_file"]).name
            if core["meta"].get("gsw_input_file") else None
        ),
        "gsw_input_sha256_file": core["meta"].get("gsw_input_sha256_file"),
        "notebook_filename": NOTEBOOK_FILENAME,
        "generated_utc": datetime.now(timezone.utc).isoformat(),
        "python_version": platform.python_version(),
    }
    manifest_path = directory / f"{label}_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return csv, manifest

def _circular_block_sample_matrix(x, B, n, block_len, rng):
    """B circular moving-block samples, each of length n."""
    x = np.asarray(x, float); N = len(x)
    n_blocks = int(np.ceil(n / block_len))
    starts = rng.integers(0, N, size=(int(B), n_blocks))
    offsets = np.arange(block_len)
    idx = (starts[:, :, None] + offsets[None, None, :]) % N
    return x[idx].reshape(int(B), -1)[:, :n]


def _recursive_forecast_weight_matrices(P_small, P_big, yv, start,
                                        H=H, minobs=MINW):
    """
    Precompute the exact linear forecast weights for the expanding small and big
    models. This makes 9,999-replication recursive bootstraps practical without
    changing the forecast protocol.
    """
    Ps, Pb, yv = _as2d(P_small), _as2d(P_big), np.asarray(yv, float)
    Tn = len(yv); eval_idx = []; Ws = []; Wb = []
    for t in range(int(start), Tn):
        if not (np.isfinite(yv[t]) and np.isfinite(Ps[t]).all() and np.isfinite(Pb[t]).all()):
            continue
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(yv[s_]) & np.isfinite(Ps[s_]).all(1) & np.isfinite(Pb[s_]).all(1)
        if m.sum() < minobs:
            continue
        js = s_[m]
        Xs = np.column_stack([np.ones(m.sum()), Ps[js]])
        Xb = np.column_stack([np.ones(m.sum()), Pb[js]])
        xs_t = np.r_[1.0, Ps[t]]
        xb_t = np.r_[1.0, Pb[t]]
        ws = np.zeros(Tn); wb = np.zeros(Tn)
        ws[js] = xs_t @ np.linalg.pinv(Xs)
        wb[js] = xb_t @ np.linalg.pinv(Xb)
        eval_idx.append(t); Ws.append(ws); Wb.append(wb)
    if not eval_idx:
        raise RuntimeError("No valid recursive forecast origins")
    return np.asarray(eval_idx, int), np.asarray(Ws, float), np.asarray(Wb, float)


def _nw_t_rows(f, nwlag=12):
    """Newey-West t statistics row by row for a 2D adjusted-loss matrix."""
    f = np.asarray(f, float)
    if f.ndim == 1:
        f = f[None, :]
    n = f.shape[1]
    mu = np.mean(f, axis=1)
    u = f - mu[:, None]
    S = np.sum(u * u, axis=1) / n
    for lag in range(1, nwlag + 1):
        w = 1 - lag / (nwlag + 1)
        S += 2 * w * np.sum(u[:, lag:] * u[:, :-lag], axis=1) / n
    se = np.sqrt(np.maximum(S, 0.0) / n)
    return np.divide(mu, se, out=np.full_like(mu, np.nan), where=se > 0)


def conditional_null_bootstrap_nested(P_small, P_big, yv, start, B=BOOTSTRAP_B,
                                      block_len=12, seed=BOOTSTRAP_SEED,
                                      H=H, minobs=MINW,
                                      batch_size=BOOTSTRAP_BATCH_SIZE):
    """
    Fast conditional-on-predictors recursive moving-block bootstrap under the
    small-model null. The expanding regressions are exactly re-run through
    precomputed linear forecast weights in every pseudo-sample.
    """
    Ps, Pb, yv = _as2d(P_small), _as2d(P_big), np.asarray(yv, float)
    obs = nested_oos_cw(Ps, Pb, yv, start=start, H=H, minobs=minobs)
    eval_idx, Ws, Wb = _recursive_forecast_weight_matrices(
        Ps, Pb, yv, start=start, H=H, minobs=minobs
    )

    # Internal identity check: precomputed weights must reproduce the observed CW.
    fs_obs = Ws @ np.nan_to_num(yv, nan=0.0)
    fb_obs = Wb @ np.nan_to_num(yv, nan=0.0)
    ye_obs = yv[eval_idx]
    f_obs = (ye_obs - fs_obs) ** 2 - (
        (ye_obs - fb_obs) ** 2 - (fs_obs - fb_obs) ** 2
    )
    cw_fast = _nw_t_rows(f_obs, nwlag=12)[0]
    if not np.isclose(cw_fast, obs["CW_t"], atol=1e-9, rtol=1e-8):
        raise AssertionError(
            f"Fast recursive weights do not reproduce CW: {cw_fast} vs {obs['CW_t']}"
        )

    # Null DGP: full-sample small-model fitted values plus block-resampled residuals.
    m_small = np.isfinite(yv) & np.isfinite(Ps).all(1)
    X = np.column_stack([np.ones(m_small.sum()), Ps[m_small]])
    beta = np.linalg.lstsq(X, yv[m_small], rcond=None)[0]
    fitted_valid = X @ beta
    resid = yv[m_small] - fitted_valid
    resid = resid - resid.mean()
    valid_idx = np.where(m_small)[0]

    rng = np.random.default_rng(seed)
    tstars = []
    remaining = int(B)
    while remaining > 0:
        bnow = min(int(batch_size), remaining)
        rstar = _circular_block_sample_matrix(
            resid, bnow, len(resid), block_len, rng
        )
        ystar = np.zeros((bnow, len(yv)), float)
        ystar[:, valid_idx] = fitted_valid[None, :] + rstar
        fs = ystar @ Ws.T
        fb = ystar @ Wb.T
        ye = ystar[:, eval_idx]
        f = (ye - fs) ** 2 - ((ye - fb) ** 2 - (fs - fb) ** 2)
        tstars.append(_nw_t_rows(f, nwlag=12))
        remaining -= bnow

    tstars = np.concatenate(tstars)
    tstars = tstars[np.isfinite(tstars)]
    p = (1 + np.sum(tstars >= obs["CW_t"])) / (1 + len(tstars)) if len(tstars) else np.nan
    return dict(
        observed_CW=obs["CW_t"],
        bootstrap_p=p,
        B_effective=len(tstars),
        block_len=block_len,
        q90=np.nanquantile(tstars, .90),
        q95=np.nanquantile(tstars, .95),
        q99=np.nanquantile(tstars, .99),
    )

# ============================================================================
# 9. UPGRADED ROBUSTNESS HELPERS (this notebook; incorporate external review)
# ============================================================================
def carry_future_decomposition(Y, taus=None, tenors=None, H=H):
    # Split rx(n)_{t->t+H} into a component KNOWN at t and the FUTURE part:
    #   rx = [n*y_n,t - (n-1)*y_{n-1,t} - y1_t]  +  [-(n-1)*(y_{n-1,t+H} - y_{n-1,t})]
    #      =              carry_t               +             future_t
    # Identity is exact (asserted in the self-test). Predicting carry is trivial
    # (it is known at t); all genuine forecasting content concerns future_t.
    taus = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    tenors = TENORS if tenors is None else tenors
    Tn = len(Y); y1 = Y[:, 0]; CARRY = {}; FUT = {}
    for n in tenors:
        ynm1 = y_at(Y, taus, n - 1); jn = taus.tolist().index(n)
        car = np.full(Tn, np.nan); fut = np.full(Tn, np.nan)
        for t in range(Tn - H):
            car[t] = n * Y[t, jn] - (n - 1) * ynm1[t] - y1[t]
            fut[t] = -(n - 1) * (ynm1[t + H] - ynm1[t])
        CARRY[n] = car; FUT[n] = fut
    return CARRY, FUT


def cieslak_povala_expanding(Y, rx_bar, pi_trend, taus=None, minobs=MINW, H=H):
    """
    Strictly real-time Cieslak-Povala-style cycle factor. This is a transparent
    economic benchmark, not a claim of exact replication of their CPI filter.
    """
    tau = np.asarray(pi_trend, float); Tn, K = Y.shape
    C = np.full((Tn, K), np.nan)
    for t in range(minobs, Tn):
        if not np.isfinite(tau[t]): continue
        s_ = np.arange(0, t)  # strictly past-only loadings
        m0 = np.isfinite(tau[s_])
        if m0.sum() < minobs: continue
        for j in range(K):
            m = m0 & np.isfinite(Y[s_, j])
            if m.sum() < minobs: continue
            X = np.column_stack([np.ones(m.sum()), tau[s_][m]])
            b = np.linalg.lstsq(X, Y[s_, j][m], rcond=None)[0]
            C[t, j] = Y[t, j] - (b[0] + b[1] * tau[t])
    c_short = C[:, 0]
    ok_long = np.isfinite(C[:, 1:]).all(1)
    c_long = np.where(ok_long, np.nanmean(C[:, 1:], axis=1), np.nan)
    Z = np.column_stack([c_short, c_long]); rb = np.asarray(rx_bar, float)
    out = np.full(Tn, np.nan)
    for t in range(minobs, Tn):
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(rb[s_]) & np.isfinite(Z[s_]).all(1)
        if m.sum() < minobs or not np.isfinite(Z[t]).all(): continue
        X = np.column_stack([np.ones(m.sum()), Z[s_][m]])
        g = np.linalg.lstsq(X, rb[s_][m], rcond=None)[0]
        out[t] = g[0] + Z[t] @ g[1:]
    return out

def oos_errors(P, yv, H=H, minobs=MINW, start=None):
    # Same protocol as oos_r2, but returns per-origin squared errors
    # (idx, e_model^2, e_bench^2) so OOS-R2 can be split by subperiod.
    P = np.asarray(P, float); P = P.reshape(-1, 1) if P.ndim == 1 else P
    yv = np.asarray(yv, float); t_start = (minobs + H) if start is None else start
    idx = []; em2 = []; eb2 = []
    for t in range(t_start, len(yv)):
        if not np.isfinite(yv[t]) or not np.isfinite(P[t]).all():
            continue
        s_ = np.arange(0, t - H + 1); m = np.isfinite(yv[s_]) & np.isfinite(P[s_]).all(1)
        if m.sum() < minobs:
            continue
        Xtr = np.column_stack([np.ones(m.sum()), P[s_][m]])
        btr = np.linalg.lstsq(Xtr, yv[s_][m], rcond=None)[0]
        idx.append(t)
        em2.append((yv[t] - (btr[0] + P[t] @ btr[1:])) ** 2)
        eb2.append((yv[t] - yv[s_][m].mean()) ** 2)
    return np.array(idx, int), np.array(em2, float), np.array(eb2, float)


def subperiod_oos_table(dates, idx, em2, eb2, edges):
    # edges: list of (label, lo_date_or_None, hi_date_or_None) with [lo, hi).
    dd = pd.DatetimeIndex(dates)[idx]
    print(f'  {"period":18s} {"n":>4s} {"OOS R2":>9s} {"share of MSPE gap":>19s}')
    total_gap = float(np.sum(eb2 - em2))
    for lab, lo, hi in edges:
        m = np.ones(len(dd), bool)
        if lo is not None: m &= dd >= pd.Timestamp(lo)
        if hi is not None: m &= dd < pd.Timestamp(hi)
        if m.sum() == 0:
            print(f'  {lab:18s} {0:>4d} {"-":>9s} {"-":>19s}'); continue
        r2 = 1 - em2[m].sum() / eb2[m].sum() if eb2[m].sum() > 0 else np.nan
        share = (eb2[m] - em2[m]).sum() / total_gap if total_gap != 0 else np.nan
        print(f'  {lab:18s} {int(m.sum()):>4d} {r2:>+9.4f} {share:>+19.2f}')


def expanding_hw_tp(Y, r, anchor_sr, taus=None, minw=MINW, freeze_at=None):
    # Fast HW-only path: expanding (kappa, sigma) + the TP_HW matrix. If
    # freeze_at=t*, parameters are FROZEN at t* for all t>t* (theta_t still moves:
    # the anchor is an exogenous input) - the 'kappa frozen in 1997' experiment.
    taus_ = np.array(MATS_L, float) if taus is None else np.asarray(taus, float)
    Tn = len(r); kk = np.full(Tn, np.nan); ss = np.full(Tn, np.nan)
    TP = np.full((Tn, len(taus_)), np.nan)
    for t in range(minw, Tn):
        if freeze_at is not None and t > freeze_at:
            k, s = kk[freeze_at], ss[freeze_at]
        else:
            k, s, _ = fit_hw(r[:t + 1], anchor_sr[:t + 1])
        kk[t], ss[t] = k, s
        for j, tau in enumerate(taus_):
            TP[t, j] = Y[t, j] - vas_yield(tau, r[t], k, anchor_sr[t], s)
    return kk, ss, TP


def rolling_hw_kappa(r, anchor_sr, win=120):
    Tn = len(r); out = np.full(Tn, np.nan)
    for t in range(win, Tn):
        k, _, _ = fit_hw(r[t - win:t + 1], anchor_sr[t - win:t + 1]); out[t] = k
    return out


def nonoverlapping_check(P, rx_n, dates, min_obs=15):
    # 12 non-overlapping annual-return series (one per calendar start month):
    # plain OLS t per series (no overlap => no NW needed). Low power, maximal
    # transparency. Returns rows (month, n, beta_on_zscored_P, t).
    P = np.asarray(P, float); yv = np.asarray(rx_n, float)
    months = pd.DatetimeIndex(dates).month; rows = []
    for m0 in range(1, 13):
        sel = np.where((months == m0) & np.isfinite(P) & np.isfinite(yv))[0]
        if len(sel) < min_obs:
            continue
        x = P[sel]; x = (x - x.mean()) / (x.std() if x.std() > 0 else 1.0); yy = yv[sel]
        X = np.column_stack([np.ones(len(x)), x]); b = np.linalg.lstsq(X, yy, rcond=None)[0]
        e = yy - X @ b
        sxx = np.sum((x - x.mean()) ** 2)
        se = np.sqrt((e @ e) / max(len(x) - 2, 1) / sxx) if sxx > 0 else np.nan
        rows.append((m0, len(x), float(b[1]), float(b[1] / se) if se and np.isfinite(se) else np.nan))
    return rows


# ============================================================================
# EXT-10: ALFRED VINTAGES / NET-OF-COST ECONOMIC VALUE / CROSS-COUNTRY L-S
# Appended verbatim after the canonical notebook-08 library. Everything below
# is past-only unless explicitly labelled otherwise, and every engine has an
# identity or invariance assertion exercised by the self-test cell.
# ============================================================================

ALFRED_SERIES = "CPIAUCSL"
G10_MODEL_PANEL_FILENAME = "g10_panel_model_objects.csv"
G10_MODEL_PANEL_SHA256 = ("1628c4eaff487f6d87cc93fe62c7af90"
                          "9850d9ceedf448d9d8321f8b8477bbf6")
COST_GRID_BP = [0.0, 1.0, 2.0, 5.0, 10.0]
LS_TOP_K = 3
LS_MIN_COUNTRIES = 6
VINTAGE_MAX_CARRY = 3      # months the slow state may be carried past last release


# ---------------------------------------------------------------------------
# A. ALFRED vintage matrix
# ---------------------------------------------------------------------------
def _read_table_any(path):
    for sep in [",", "\t", ";"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    raise ValueError(f"Could not parse {path} as a delimited table")


def parse_alfred_vintages(path, series=ALFRED_SERIES):
    """Parse an ALFRED 'all vintages' wide file into an observation x vintage
    matrix. Columns are recognised as SERIES_YYYYMMDD or SERIES_YYYY-MM-DD.
    Observation dates are normalised to month-end. Values absent in a given
    vintage stay NaN (no filling across vintages)."""
    import re as _re
    df = _read_table_any(path)
    dcol = None
    for c in df.columns:
        if "date" in str(c).lower():
            dcol = c
            break
    if dcol is None:
        dcol = df.columns[0]
    vint = {}
    for c in df.columns:
        s = str(c).strip()
        m = _re.match(rf"^{series}[_ ](\d{{8}})$", s) or \
            _re.match(rf"^{series}[_ ](\d{{4}}-\d{{2}}-\d{{2}})$", s)
        if m:
            vint[pd.Timestamp(m.group(1))] = c
    if not vint:
        raise ValueError(f"No {series}_<vintage> columns found in {path}")
    obs = pd.to_datetime(df[dcol]) + pd.offsets.MonthEnd(0)
    M = pd.DataFrame(
        {v: pd.to_numeric(df[c], errors="coerce").to_numpy() for v, c in sorted(vint.items())},
        index=obs,
    ).sort_index()
    M.columns = pd.DatetimeIndex(M.columns)
    M = M.groupby(level=0).last()
    return M


def vintage_asof_series(M, t):
    """CPI level series as known at month-end t: latest vintage dated <= t,
    restricted to the observations actually present in that vintage."""
    cols = M.columns[M.columns <= pd.Timestamp(t)]
    if len(cols) == 0:
        return None, None
    v = cols[-1]
    return M[v].dropna(), v


def vintage_hw_tp(dates, r, Y, taus, M, pi_hl=PI_HL, rstar_hl=RSTAR_HL,
                  initialization=PAPER_ANCHOR_INIT, minw=MINW,
                  mode="asof", max_carry=VINTAGE_MAX_CARRY):
    """Real-time vintage anchor -> expanding HW -> per-tenor TP matrix.

    mode='asof'     : at each origin t the FULL anchor history is rebuilt from
                      the vintage available at t (standard real-time practice).
    mode='diagonal' : theta(s) is frozen at its first-usable value (vintage at s)
                      and never revised; the expanding fit runs on that path.

    The slow state is carried forward at most `max_carry` months past the last
    observation in the vintage (release gap), and the staleness is recorded.
    Returns dict(TP, kappa, sigma, staleness, n_skipped).
    """
    dates = pd.DatetimeIndex(dates)
    Tn, K = len(dates), len(taus)
    TP = np.full((Tn, K), np.nan)
    kk = np.full(Tn, np.nan)
    ss = np.full(Tn, np.nan)
    stale = np.full(Tn, np.nan)
    skipped = 0

    def _theta_from_cpi(cpi_arr):
        res = build_anchor(dates, r, cpi_arr, pi_hl=pi_hl, rstar_hl=rstar_hl,
                           initialization=initialization)
        return np.array(res[0], dtype=float, copy=True)

    if mode == "diagonal":
        theta_diag = np.full(Tn, np.nan)
        for t in range(Tn):
            s, _v = vintage_asof_series(M, dates[t])
            if s is None:
                continue
            cpi_arr = s.reindex(dates).to_numpy(float)
            cpi_fin = np.where(np.isfinite(cpi_arr[: t + 1]))[0]
            if len(cpi_fin) == 0:
                continue
            L = int(cpi_fin[-1])          # last released observation
            if t - L > max_carry:
                continue
            th = _theta_from_cpi(cpi_arr)
            th[L + 1:] = np.nan           # pandas ewm glides over trailing
                                          # NaNs; enforce release boundary
            fin = np.where(np.isfinite(th[: L + 1]))[0]
            if len(fin) == 0:
                continue
            Lth = int(fin[-1])            # last finite theta at/before L
            theta_diag[t] = th[Lth]
            stale[t] = t - Lth
        for t in range(minw, Tn):
            th_hist = theta_diag[: t + 1]
            m = np.isfinite(th_hist) & np.isfinite(r[: t + 1])
            if m.sum() < minw or not np.isfinite(theta_diag[t]):
                continue
            k, s_, _ = fit_hw(r[: t + 1][m], th_hist[m])
            kk[t], ss[t] = k, s_
            for j, tau in enumerate(taus):
                TP[t, j] = Y[t, j] - vas_yield(tau, r[t], k, theta_diag[t], s_)
        return dict(TP=TP, kappa=kk, sigma=ss, staleness=stale, n_skipped=skipped)

    # asof mode
    for t in range(minw, Tn):
        s, _v = vintage_asof_series(M, dates[t])
        if s is None:
            skipped += 1
            continue
        cpi_arr = s.reindex(dates).to_numpy(float)
        cpi_fin = np.where(np.isfinite(cpi_arr[: t + 1]))[0]
        if len(cpi_fin) == 0:
            skipped += 1
            continue
        L = int(cpi_fin[-1])              # last released observation
        if t - L > max_carry:
            skipped += 1
            continue
        th = _theta_from_cpi(cpi_arr)
        th[L + 1:] = np.nan               # enforce release boundary against
                                          # pandas ewm NaN-glide
        fin = np.where(np.isfinite(th[: L + 1]))[0]
        if len(fin) == 0:
            skipped += 1
            continue
        Lth = int(fin[-1])
        th_eff = th.copy()
        if Lth < t:
            th_eff[Lth + 1: t + 1] = th[Lth]
        stale[t] = t - Lth
        k, s_, _ = fit_hw(r[: t + 1], th_eff[: t + 1])
        kk[t], ss[t] = k, s_
        for j, tau in enumerate(taus):
            TP[t, j] = Y[t, j] - vas_yield(tau, r[t], k, th_eff[t], s_)
    return dict(TP=TP, kappa=kk, sigma=ss, staleness=stale, n_skipped=skipped)


# ---------------------------------------------------------------------------
# B. Net-of-cost duration timing
# ---------------------------------------------------------------------------
def expanding_forecast(P, yv, H=H, minobs=MINW, start=None):
    """Store the recursive forecast f_t used by oos_r2 (same conventions)."""
    P = np.asarray(P, float)
    P = P.reshape(-1, 1) if P.ndim == 1 else P
    yv = np.asarray(yv, float)
    t0 = (minobs + H) if start is None else start
    f = np.full(len(yv), np.nan)
    for t in range(t0, len(yv)):
        if not np.isfinite(P[t]).all():
            continue
        s_ = np.arange(0, t - H + 1)
        m = np.isfinite(yv[s_]) & np.isfinite(P[s_]).all(1)
        if m.sum() < minobs:
            continue
        X = np.column_stack([np.ones(m.sum()), P[s_][m]])
        b = np.linalg.lstsq(X, yv[s_][m], rcond=None)[0]
        f[t] = b[0] + P[t] @ b[1:]
    return f


def past_only_stats(x, minobs=24):
    """Expanding past-only mean and sd of a series (uses s <= t)."""
    x = np.asarray(x, float)
    mu = np.full(len(x), np.nan)
    sd = np.full(len(x), np.nan)
    for t in range(len(x)):
        h = x[: t + 1]
        h = h[np.isfinite(h)]
        if len(h) >= minobs:
            mu[t] = h.mean()
            sd[t] = h.std(ddof=1)
    return mu, sd


def timing_weights(f, rx_realized, rule, cap=2.0, H=H, minobs=24):
    """Map recursive 12m forecasts into monthly duration weights.
    'static' -> 1; 'sign' -> sign(f); 'linear' -> f / (2*sd_past(realized rx)),
    capped; 'zscore' -> past-only z of f, capped. All inputs at t use s<=t
    information only (realized rx enters only through s <= t-H)."""
    f = np.asarray(f, float)
    n = len(f)
    if rule == "static":
        w = np.where(np.isfinite(f), 1.0, np.nan)
        return w
    if rule == "sign":
        return np.where(np.isfinite(f), np.sign(f), np.nan)
    if rule == "linear":
        rx = np.asarray(rx_realized, float)
        sd = np.full(n, np.nan)
        for t in range(n):
            h = rx[: max(t - H + 1, 0)]
            h = h[np.isfinite(h)]
            if len(h) >= minobs:
                sd[t] = h.std(ddof=1)
        w = f / (2.0 * sd)
        return np.clip(w, -cap, cap)
    if rule == "zscore":
        mu, sd = past_only_stats(f, minobs=minobs)
        w = (f - mu) / sd
        return np.clip(w, -cap, cap)
    raise ValueError(rule)


def run_timing_strategy(w, ex1m, cost_bp=0.0, skip=0):
    """Apply weight w_t to the asset's NEXT-month excess return (t+1+skip).
    Cost = cost_bp * |w_t - w_prev| per unit duration-notional round trip.
    Returns aligned arrays (gross, net, turnover) on the valid window."""
    w = np.asarray(w, float)
    ex = np.asarray(ex1m, float)
    n = len(w)
    gross = np.full(n, np.nan)
    net = np.full(n, np.nan)
    to = np.full(n, np.nan)
    prev = np.nan
    for t in range(n - 1 - skip):
        if not np.isfinite(w[t]) or not np.isfinite(ex[t + 1 + skip]):
            continue
        g = w[t] * ex[t + 1 + skip]
        dv = abs(w[t] - prev) if np.isfinite(prev) else abs(w[t])
        c = (cost_bp / 1e4) * dv
        gross[t + 1 + skip] = g
        net[t + 1 + skip] = g - c
        to[t + 1 + skip] = dv
        prev = w[t]
    return gross, net, to


def strategy_stats(ret_m, gamma=(3.0, 5.0)):
    r = np.asarray(ret_m, float)
    r = r[np.isfinite(r)]
    if len(r) < 24:
        return None
    ann_mu = 12.0 * r.mean()
    ann_sd = np.sqrt(12.0) * r.std(ddof=1)
    sr = ann_mu / ann_sd if ann_sd > 0 else np.nan
    cum = np.cumsum(r)
    mdd = float(np.max(np.maximum.accumulate(cum) - cum))
    out = dict(n=len(r), ann_mean=ann_mu, ann_vol=ann_sd, sharpe=sr, mdd=mdd)
    for g in gamma:
        out[f"CE_g{int(g)}"] = ann_mu - 0.5 * g * ann_sd ** 2
    return out


def breakeven_cost(w, ex1m, target_sharpe, grid_bp=(0, 1, 2, 5, 10, 20, 40, 80), skip=0):
    xs, ys = [], []
    for c in grid_bp:
        _, net, _ = run_timing_strategy(w, ex1m, cost_bp=c, skip=skip)
        st = strategy_stats(net)
        if st is None:
            return np.nan
        xs.append(c)
        ys.append(st["sharpe"])
    xs = np.array(xs, float)
    ys = np.array(ys, float)
    if ys[0] <= target_sharpe:
        return 0.0
    below = np.where(ys <= target_sharpe)[0]
    if len(below) == 0:
        return float(xs[-1])
    j = below[0]
    x0, x1, y0, y1 = xs[j - 1], xs[j], ys[j - 1], ys[j]
    return float(x0 + (y0 - target_sharpe) * (x1 - x0) / (y0 - y1))


# ---------------------------------------------------------------------------
# C. Cross-country long-short on the frozen G10 model panel
# ---------------------------------------------------------------------------
def load_g10_model_panel(filename=G10_MODEL_PANEL_FILENAME,
                         expected_sha=G10_MODEL_PANEL_SHA256):
    path, h = _resolve_local_file(filename, expected_sha256=None)
    try:
        if expected_sha and h != expected_sha:
            warnings.warn(f"{filename}: SHA-256 {h[:12]}... differs from the "
                          f"notebook-09 archive {expected_sha[:12]}... "
                          "(acceptable if the panel was rebuilt; provenance noted)")
        else:
            print(f"G10 model panel hash verified: {h[:16]}...")
    except Exception as e:
        warnings.warn(f"hash check skipped: {e}")
    df = pd.read_csv(path)
    need = {"date", "country", "tp_z_past", "rx"}
    if not need.issubset(df.columns):
        raise ValueError(f"{filename} missing columns {need - set(df.columns)}")
    df["date"] = pd.to_datetime(df["date"])
    return df


def cross_sectional_ls(panel, signal="tp_z_past", target="rx",
                       k=LS_TOP_K, min_countries=LS_MIN_COUNTRIES):
    """Monthly-formed, duration-balanced long-short of 12m excess returns.
    Long the k highest past-only signals, short the k lowest, equal weights.
    Legs are local-currency excess returns over local funding, so the spread
    is FX-hedged by construction. Returns a per-date DataFrame."""
    rows = []
    prevL, prevS = None, None
    for date, g in panel.groupby("date"):
        g = g[["country", signal, target]].dropna()
        if len(g) < max(min_countries, 2 * k):
            prevL, prevS = None, None
            continue
        g = g.sort_values(signal)
        S = set(g["country"].iloc[:k])
        L = set(g["country"].iloc[-k:])
        spread = g[target].iloc[-k:].mean() - g[target].iloc[:k].mean()
        sig_spread = g[signal].iloc[-k:].mean() - g[signal].iloc[:k].mean()
        if prevL is not None:
            events = len(L - prevL) + len(S - prevS)
        else:
            events = np.nan
        rows.append(dict(date=date, spread=spread, sig_spread=sig_spread,
                         n_countries=len(g), events=events))
        prevL, prevS = L, S
    return pd.DataFrame(rows).set_index("date").sort_index()


def ls_summary(ls, k=LS_TOP_K, cost_grid=COST_GRID_BP, nw_lag=12):
    sp = ls["spread"].to_numpy(float)
    b, se, _, n = nw_ols(sp, np.empty((len(sp), 0)), nw_lag)
    mean_ann = float(np.nanmean(sp))
    tstat = float(b[0] / se[0])
    per_sd = mean_ann / float(np.nanmean(ls["sig_spread"]))
    # 12 non-overlapping calendar tracks (honest Sharpe under 12m overlap)
    months = ls.index.month
    track_sr = []
    for m0 in range(1, 13):
        x = sp[months == m0]
        x = x[np.isfinite(x)]
        if len(x) >= 10 and x.std(ddof=1) > 0:
            track_sr.append(x.mean() / x.std(ddof=1))
    ev = ls["events"].to_numpy(float)
    ev_m = float(np.nanmean(ev))
    out = dict(months=int(n), gross_ann_mean=mean_ann, NW_t=tstat,
               per_signal_SD=per_sd, median_track_SR=float(np.median(track_sr)),
               tracks_positive=f"{sum(1 for s in track_sr if s > 0)}/{len(track_sr)}",
               events_per_month=ev_m)
    for c in cost_grid:
        drag = 12.0 * (c / 1e4) * ev_m / k
        out[f"net_ann_mean_c{int(c)}bp"] = mean_ann - drag
    # breakeven: gross mean fully consumed
    out["breakeven_bp"] = (mean_ann * 1e4 * k / (12.0 * ev_m)) if ev_m > 0 else np.nan
    return out


def nw_ols(y, X, lag, add_intercept=True):
    """NW mean/regression helper (panel-09 convention)."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    if X.ndim == 1:
        X = X[:, None]
    m = np.isfinite(y) & (np.isfinite(X).all(axis=1) if X.size else np.ones(len(y), bool))
    y = y[m]
    X = X[m] if X.size else np.empty((len(y), 0))
    if add_intercept:
        X = np.column_stack([np.ones(len(y)), X]) if X.size else np.ones((len(y), 1))
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    XtX = np.linalg.pinv(X.T @ X)
    Xe = X * e[:, None]
    S = Xe.T @ Xe
    for l in range(1, lag + 1):
        w_ = 1 - l / (lag + 1)
        G = Xe[l:].T @ Xe[:-l]
        S = S + w_ * (G + G.T)
    V = XtX @ S @ XtX
    se = np.sqrt(np.maximum(np.diag(V), 0))
    r2 = 1 - (e @ e) / max(((y - y.mean()) ** 2).sum(), 1e-30)
    return b, se, r2, int(len(y))


def hw_only_tp(Y, r, state, taus, minw=MINW):
    """Fast HW-only expanding fit against a given state path; per-tenor TP."""
    Tn = len(r)
    TP = np.full((Tn, len(taus)), np.nan)
    for t in range(minw, Tn):
        m = np.isfinite(state[: t + 1]) & np.isfinite(r[: t + 1])
        if m.sum() < minw or not np.isfinite(state[t]):
            continue
        k, s_, _ = fit_hw(r[: t + 1][m], state[: t + 1][m])
        for j, tau in enumerate(taus):
            TP[t, j] = Y[t, j] - vas_yield(tau, r[t], k, state[t], s_)
    return TP


## Self-test (offline, synthetic) — identities and look-ahead invariance

In [10]:
# SELF-TEST (offline, synthetic). Hard asserts on identities and look-ahead
# invariance; nothing here touches the network or the frozen inputs.
def _make_world(T=420, seed=11):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("1980-01-31", periods=T, freq="ME")
    taus = np.array(MATS_L, float)
    rstar = np.clip(0.02 + 0.003*np.cumsum(rng.standard_normal(T))/np.sqrt(T), -0.005, 0.045)
    pim = 0.028 + 0.018*np.sin(np.linspace(0, 2.5*np.pi, T)); pi = np.empty(T); pi[0] = 0.035
    for t in range(1, T): pi[t] = pi[t-1] + 0.12*(pim[t]-pi[t-1]) + 0.003*rng.standard_normal()
    pi = np.clip(pi, 0.002, 0.09); theta = rstar + pi
    r = np.empty(T); r[0] = theta[0]; KAP, SIG = 0.16, 0.008
    for t in range(1, T): r[t] = r[t-1] + KAP/12*(theta[t]-r[t-1]) + SIG/np.sqrt(12)*rng.standard_normal()
    r = np.clip(r, 5e-4, 0.13); cpi = np.empty(T); cpi[0] = 100.0
    for t in range(1, T): cpi[t] = cpi[t-1]*(1 + pi[t]/12 + 8e-4*rng.standard_normal())
    tp = np.empty(T); tp[0] = 0.012
    for t in range(1, T): tp[t] = 0.012 + 0.95*(tp[t-1]-0.012) + 0.0022*rng.standard_normal()
    Y = np.full((T, len(taus)), np.nan)
    for t in range(T):
        for j, tau in enumerate(taus):
            Y[t, j] = vas_yield(tau, r[t], KAP, theta[t], SIG) + tp[t]*(tau/10.0)**0.9 + 4e-4*rng.standard_normal()
    return dates, np.clip(Y, 2e-4, 0.16), r, cpi

dts_, Y_, r_, cpi_ = _make_world()
taus_ = np.array(MATS_L, float)

# ---- A. vintage engine -----------------------------------------------------
# synthetic ALFRED matrix: one vintage per month, released day 15 of m+1,
# containing observations through m (zero revisions in the base case)
vints = dts_ + pd.offsets.MonthBegin(1) + pd.Timedelta(days=14)
M_ = pd.DataFrame(index=dts_, columns=pd.DatetimeIndex(vints), dtype=float)
for j, v in enumerate(vints):
    M_.iloc[: j + 1, j] = cpi_[: j + 1]

vin = vintage_hw_tp(dts_, r_, Y_, taus_, M_, initialization="post_ewma_bfill")

# reference under zero revisions: final anchor known through t-1, carried 1m
anc_full = np.asarray(build_anchor(dts_, r_, cpi_, initialization="post_ewma_bfill")[0], float)
TPref = np.full_like(vin["TP"], np.nan)
for t in range(MINW, len(r_)):
    th = anc_full.copy(); th[t] = anc_full[t - 1]           # last release = t-1
    k, s_, _ = fit_hw(r_[: t + 1], th[: t + 1])
    for j, tau in enumerate(taus_):
        TPref[t, j] = Y_[t, j] - vas_yield(tau, r_[t], k, th[t], s_)
d0 = np.nanmax(np.abs(vin["TP"] - TPref))
assert d0 < 1e-10, f"zero-revision identity failed: {d0:.2e}"
assert np.nanmax(vin["staleness"]) <= 1.0

# look-ahead invariance: perturb vintages dated AFTER a cutoff with a
# row-dependent factor (a constant level scale would cancel in YoY inflation
# and make the test vacuous)
cut = dts_[300]
fac_ = (1.0 + 0.02 * (np.arange(len(dts_)) % 24) / 24.0)[:, None]
M2_ = M_.copy(); late = M2_.columns > (cut + pd.Timedelta(days=20))
M2_.loc[:, late] = M2_.loc[:, late].to_numpy() * fac_
vin2 = vintage_hw_tp(dts_, r_, Y_, taus_, M2_, initialization="post_ewma_bfill")
d1 = np.nanmax(np.abs(vin["TP"][:300] - vin2["TP"][:300]))
assert d1 < 1e-12, f"look-ahead invariance failed: {d1:.2e}"
d1b = np.nanmax(np.abs(vin["TP"][305:] - vin2["TP"][305:]))
assert d1b > 1e-7, "perturbation was vacuous: post-cutoff TP unchanged"
# revisions must matter when they touch the past (row-dependent factor again)
M3_ = M_.copy()
M3_.iloc[:, :150] = M3_.iloc[:, :150].to_numpy() * fac_
vin3 = vintage_hw_tp(dts_, r_, Y_, taus_, M3_, initialization="post_ewma_bfill")
assert np.nanmax(np.abs(vin["TP"] - vin3["TP"])) > 1e-6
vdiag = vintage_hw_tp(dts_, r_, Y_, taus_, M_, mode="diagonal")
assert np.isfinite(vdiag["TP"][:, -1]).sum() > 100
print(f"A. vintage engine: zero-revision identity {d0:.1e} | invariance {d1:.1e} | diagonal OK")

# ---- B. timing engine --------------------------------------------------------
RX_, rxb_ = excess_returns_fb(Y_, taus=taus_)
RET_, _ = monthly_returns(Y_, r_, taus=taus_)
ex1m_ = RET_[:, -1] - r_ / 12.0
f_ = expanding_forecast(TPref[:, -1], RX_[10])
o_chk, n_chk = oos_r2(TPref[:, -1], RX_[10])
# oos_r2 counts only origins with a realized target; the strategy path also
# forecasts the last H months (traded, not yet realized)
ov_ = np.isfinite(f_) & np.isfinite(RX_[10])
assert ov_.sum() == n_chk, "forecast/oos_r2 overlap mismatch"
assert (np.isfinite(f_) & ~np.isfinite(RX_[10])).sum() == H, "tail forecasts != H"
w_lin = timing_weights(f_, RX_[10], "linear")
assert np.nanmax(np.abs(w_lin)) <= 2.0 + 1e-12
g0, n0, to0 = run_timing_strategy(w_lin, ex1m_, cost_bp=0.0)
assert np.nanmax(np.abs(g0 - n0)) < 1e-15, "c=0 must give net==gross"
w_st = timing_weights(f_, RX_[10], "static")
_, _, tos = run_timing_strategy(w_st, ex1m_, cost_bp=5.0)
fin = np.where(np.isfinite(tos))[0]
assert np.nanmax(tos[fin[1]:]) < 1e-15, "static weights must have zero turnover after entry"
w_toy = np.array([np.nan]*5 + [1.0, -1.0, 1.0, 1.0])
_, _, tt = run_timing_strategy(w_toy, np.ones(9)*0.01, cost_bp=0.0)
assert abs(np.nansum(tt) - (1 + 2 + 2 + 0)) < 1e-12, "toy turnover identity failed"
st_g = strategy_stats(g0)
be = breakeven_cost(w_lin, ex1m_, target_sharpe=strategy_stats(
    run_timing_strategy(w_st, ex1m_, 0.0)[0])["sharpe"])
print(f"B. timing engine: forecasts n={n_chk} | gross SR {st_g['sharpe']:.2f} | breakeven grid OK ({be:.1f}bp)")

# ---- C. cross-sectional long-short ------------------------------------------
rngc = np.random.default_rng(5)
Tp, Nc = 240, 8
beta_true = 0.02
zs = rngc.standard_normal((Tp, Nc))
common = 0.03 * rngc.standard_normal(Tp)
rows = []
pdates = pd.date_range("2000-01-31", periods=Tp, freq="ME")
for t in range(Tp):
    for i in range(Nc):
        rows.append(dict(date=pdates[t], country=f"C{i}",
                         tp_z_past=zs[t, i],
                         rx=beta_true*zs[t, i] + common[t] + 0.01*rngc.standard_normal()))
pan_ = pd.DataFrame(rows)
ls_ = cross_sectional_ls(pan_, k=3, min_countries=6)
summ_ = ls_summary(ls_, k=3)
assert abs(summ_["per_signal_SD"] - beta_true) < 0.006, summ_["per_signal_SD"]
assert summ_["NW_t"] > 3.0
pan_c = pan_.copy(); pan_c["tp_z_past"] = pan_c.groupby("country")["tp_z_past"].transform(lambda s: np.arange(len(s)) % 1 + float(s.name[-1]))
ls_c = cross_sectional_ls(pan_c, k=3, min_countries=6)
assert np.nanmax(ls_c["events"].to_numpy()[1:]) == 0, "constant ranks must give zero churn"
print(f"C. long-short: per-SD {summ_['per_signal_SD']:.4f} vs true {beta_true} | NW t {summ_['NW_t']:.1f} | churn identity OK")

print("SELF-TEST OK - all identities and invariances hold.")


/tmp/ipykernel_576/1536259769.py:329: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  cpi_yoy = pd.Series(cpi_idx, index=dates).pct_change(12)


A. vintage engine: zero-revision identity 0.0e+00 | invariance 0.0e+00 | diagonal OK
B. timing engine: forecasts n=157 | gross SR 0.38 | breakeven grid OK (79.6bp)
C. long-short: per-SD 0.0200 vs true 0.02 | NW t 35.5 | churn identity OK
SELF-TEST OK - all identities and invariances hold.


## Load frozen inputs
Baseline sanity: the canonical basket row must reproduce Table 2 (0.0163 / CW 1.366) — the
chain-of-custody check that this notebook sits on the same stack as the paper.

If an input is missing, the discovery cell above opens a file-upload dialog on Colab (press Cancel to skip one) and stages anything it finds inside an unzipped bundle into the working directory. Each section below still skips cleanly when its own input is absent, so it is fine to upload only what a given run needs.


In [11]:
US_OK = ALFRED_OK = PANEL_OK = False
core = None; M_VINT = None; PANEL = None
try:
    core = build_us_core(curve_mode="gs_cmt_snapshot", verbose=True)
    core["cpi_idx"] = np.asarray(core.get("meta", {}).get("cpi_idx",
                        core.get("cpi_idx")), float)
    dates = core["dates"]; Y = core["Y"]; r = core["r"]; taus = core["taus"]
    RX = core["RX"]; rx_bar = core["rx_bar"]; rx10 = RX[10]
    TPHW = core["TPHW"]; tp10 = TPHW[:, -1]
    tp_bar = np.where(np.isfinite(TPHW[:, 1:]).all(1), TPHW[:, 1:].mean(1), np.nan)
    y10 = Y[:, -1]
    t0 = first_valid_start(tp_bar, rx_bar)
    o, n = oos_r2(tp_bar, rx_bar, start=t0)
    _, cw, p, _ = clark_west(tp_bar, rx_bar, start=t0)
    print(f"\nbaseline sanity (Table 2): OOS {o:+.4f} CW {cw:.3f} n {n}"
          f"  -> expected +0.0163 / 1.366 / 336")
    US_OK = True
except Exception as e:
    print("US snapshot unavailable ->", repr(e))

try:
    hits = []
    for d in CANDIDATE_DIRS:
        for p in glob.glob(str(Path(d) / "*.csv")) + glob.glob(str(Path(d) / "*.txt")):
            try:
                head = pd.read_csv(p, nrows=1)
                if any(str(c).startswith("CPIAUCSL_") for c in head.columns):
                    hits.append(p)
            except Exception:
                pass
    if not hits:
        raise FileNotFoundError("no file with CPIAUCSL_<vintage> columns found")
    M_VINT = parse_alfred_vintages(hits[0])
    print(f"\nALFRED vintages: {hits[0]} | {M_VINT.shape[1]} vintages "
          f"{M_VINT.columns[0].date()} -> {M_VINT.columns[-1].date()} | "
          f"obs {M_VINT.index[0].date()} -> {M_VINT.index[-1].date()}")
    ALFRED_OK = True
except Exception as e:
    print("ALFRED vintage file unavailable ->", repr(e))

try:
    PANEL = load_g10_model_panel()
    print(f"\nG10 model panel: {PANEL['country'].nunique()} countries, {len(PANEL)} rows")
    PANEL_OK = True
except Exception as e:
    print("G10 model panel unavailable ->", repr(e))

/tmp/ipykernel_576/1536259769.py:159: UserWarning: Parsed DataFrame hash differs from the archived pandas hash. The byte-level file hash is valid; this can reflect pandas parsing/version details.
  warnings.warn(


Assembled 599 months: 1976-06-30 -> 2026-05-31
  curve: Frozen FRED GS constant-maturity snapshot (par-equivalent) | r*: proxy: EWMA real short (hl=120m) | TP ref: THREEFYTP10
  target interpretation: CMT-based Fama-Bliss-style | anchor init: post_ewma_bfill

baseline sanity (Table 2): OOS +0.0163 CW 1.366 n 336  -> expected +0.0163 / 1.366 / 336

ALFRED vintages: CPIAUCSL_all_vintages.csv | 669 vintages 1972-07-21 -> 2026-09-11 | obs 1947-01-31 -> 2026-08-31
G10 model panel hash verified: 1628c4eaff487f6d...

G10 model panel: 10 countries, 5850 rows


## A. Full-vintage real-time anchor
Four anchors, identical downstream pipeline: **canonical** (final data), **release-lag** (§7 proxy:
CPI shifted one month), **vintage as-of** (at each origin the whole anchor history is rebuilt from
the vintage available that month — standard real-time practice), **vintage diagonal** (each month's
anchor frozen at its first-usable value, never revised). Targets are market data and revision-free.

In [12]:
if not (US_OK and ALFRED_OK):
    print("(skipped - requires US snapshot and ALFRED vintage file)")
else:
    variants = {}
    variants["canonical (final data)"] = TPHW
    cpi_lag = pd.Series(core["cpi_idx"], index=dates).shift(1).to_numpy(float)
    res = build_anchor(dates, r, cpi_lag, initialization=PAPER_ANCHOR_INIT)
    variants["release-lag (CPI t-1)"] = hw_only_tp(Y, r, np.asarray(res[0], float), taus)
    vin_a = vintage_hw_tp(dates, r, Y, taus, M_VINT, initialization=PAPER_ANCHOR_INIT, mode="asof")
    variants["vintage as-of"] = vin_a["TP"]
    vin_d = vintage_hw_tp(dates, r, Y, taus, M_VINT, initialization=PAPER_ANCHOR_INIT, mode="diagonal")
    variants["vintage diagonal"] = vin_d["TP"]
    st_ = int(np.nanmax(vin_a["staleness"])) if np.isfinite(vin_a["staleness"]).any() else -1
    print(f"as-of staleness max {st_} months | skipped origins {vin_a['n_skipped']}\n")
    print(f"{'anchor':26s} {'basket OOS':>11s} {'CW':>6s} {'10Y OOS':>9s} {'CW':>6s} {'nested CW':>10s} {'n':>5s}")
    for nm, TPv in variants.items():
        tbv = np.where(np.isfinite(TPv[:, 1:]).all(1), TPv[:, 1:].mean(1), np.nan)
        stv = max(t0, first_valid_start(tbv, rx_bar) or t0)
        ob, nb_ = oos_r2(tbv, rx_bar, start=stv); _, cb, _, _ = clark_west(tbv, rx_bar, start=stv)
        o1, _ = oos_r2(TPv[:, -1], rx10, start=stv); _, c1, _, _ = clark_west(TPv[:, -1], rx10, start=stv)
        ns = nested_oos_cw(y10, np.column_stack([y10, TPv[:, -1]]), rx10, start=stv)
        print(f"{nm:26s} {ob:>+11.4f} {cb:>6.2f} {o1:>+9.4f} {c1:>6.2f} {ns['CW_t']:>10.2f} {nb_:>5d}")
    # pre-registered verdict
    tb_can = np.where(np.isfinite(variants['canonical (final data)'][:, 1:]).all(1),
                      variants['canonical (final data)'][:, 1:].mean(1), np.nan)
    tb_vin = np.where(np.isfinite(variants['vintage as-of'][:, 1:]).all(1),
                      variants['vintage as-of'][:, 1:].mean(1), np.nan)
    stc = max(t0, first_valid_start(tb_vin, rx_bar) or t0)
    oc, _ = oos_r2(tb_can, rx_bar, start=stc); ov, _ = oos_r2(tb_vin, rx_bar, start=stc)
    nsv = nested_oos_cw(y10, np.column_stack([y10, variants['vintage as-of'][:, -1]]), rx10, start=stc)
    ok = (abs(oc - ov) <= 0.010) and (nsv["CW_t"] >= 1.645)
    print("\nVERDICT (pre-registered):",
          "vintage-robust -> drop the US pseudo-OOS asterisk; report the as-of row in Section 7."
          if ok else
          f"degradation beyond threshold (dBasket {ov-oc:+.4f}, nested CW {nsv['CW_t']:.2f}) -> report as quantified limitation.")

/tmp/ipykernel_576/1536259769.py:329: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  cpi_yoy = pd.Series(cpi_idx, index=dates).pct_change(12)


as-of staleness max 2 months | skipped origins 0

anchor                      basket OOS     CW   10Y OOS     CW  nested CW     n
canonical (final data)         +0.0163   1.37   +0.0985   2.35       3.15   336
release-lag (CPI t-1)          +0.0145   1.37   +0.0967   2.37       3.13   336
vintage as-of                  +0.0147   1.13   +0.0986   2.18       0.84   336
vintage diagonal               +0.0011   1.19   +0.0472   1.90       1.10   336

VERDICT (pre-registered): degradation beyond threshold (dBasket -0.0015, nested CW 0.84) -> report as quantified limitation.


## B. Net-of-cost duration timing (US 10Y, CMT branch for §7 continuity)
Monthly weights from the recursive 12m forecast, applied to next month's 10Y excess return.
Costs per unit duration-notional round trip; grid {0,1,2,5,10}bp. `skip=1` trades one extra month
late. All weights are past-only; caps at ±2.

In [13]:
if not US_OK:
    print("(skipped - requires US snapshot)")
else:
    RET_pt, _ = monthly_returns(Y, r, taus=taus)
    ex1m = RET_pt[:, -1] - r / 12.0
    f10 = expanding_forecast(tp10, rx10)
    rules = ["static", "sign", "linear", "zscore"]
    W = {rl: timing_weights(f10, rx10, rl) for rl in rules}
    stats0 = {rl: strategy_stats(run_timing_strategy(W[rl], ex1m, 0.0)[0]) for rl in rules}
    base_sr = stats0["static"]["sharpe"]; base_ce5 = stats0["static"]["CE_g5"]
    print(f"{'rule':8s} {'annMean':>8s} {'annVol':>7s} {'SR':>6s} {'MDD':>6s} "
          f"{'dCE(g5)':>8s} | net SR @ " + " / ".join(f"{int(c)}bp" for c in COST_GRID_BP)
          + f" | {'breakeven':>9s}")
    for rl in rules:
        s0 = stats0[rl]
        nets = []
        for c in COST_GRID_BP:
            _, netc, _ = run_timing_strategy(W[rl], ex1m, cost_bp=c)
            nets.append(strategy_stats(netc)["sharpe"])
        be = breakeven_cost(W[rl], ex1m, target_sharpe=base_sr) if rl != "static" else np.nan
        print(f"{rl:8s} {s0['ann_mean']*100:>7.2f}% {s0['ann_vol']*100:>6.2f}% "
              f"{s0['sharpe']:>6.2f} {s0['mdd']*100:>5.1f}% {s0['CE_g5']-base_ce5:>+8.4f} | "
              + " / ".join(f"{v:.2f}" for v in nets) + f" | {be:>7.1f}bp")
    for rl in ["linear", "zscore"]:
        g1, _, _ = run_timing_strategy(W[rl], ex1m, 0.0, skip=1)
        print(f"skip-month {rl}: gross SR {strategy_stats(g1)['sharpe']:.2f}")
    _, net2, _ = run_timing_strategy(W["linear"], ex1m, 2.0)
    ok = (strategy_stats(net2)["sharpe"] >= base_sr) and \
         (strategy_stats(net2)["CE_g5"] > base_ce5)
    print("\nVERDICT (pre-registered):",
          "net-of-cost line is publishable at 2bp (linear rule)." if ok else
          "net-of-cost gain not established at 2bp -> keep gross-only framing, report breakeven.")

rule      annMean  annVol     SR    MDD  dCE(g5) | net SR @ 0bp / 1bp / 2bp / 5bp / 10bp | breakeven
static      3.16%   7.45%   0.42  38.3%  +0.0000 | 0.42 / 0.42 / 0.42 / 0.42 / 0.42 |     nanbp
sign        1.57%   7.49%   0.21  28.1%  -0.0160 | 0.21 / 0.21 / 0.20 / 0.19 / 0.18 |     0.0bp
linear      0.38%   0.75%   0.51   2.2%  -0.0140 | 0.51 / 0.51 / 0.50 / 0.49 / 0.46 |    18.0bp
zscore      3.33%   6.72%   0.50  19.8%  +0.0042 | 0.50 / 0.49 / 0.49 / 0.47 / 0.45 |    15.0bp
skip-month linear: gross SR 0.77
skip-month zscore: gross SR 0.76

VERDICT (pre-registered): net-of-cost gain not established at 2bp -> keep gross-only framing, report breakeven.


## C. Cross-country long–short on the frozen panel
Top-3 / bottom-3 by `tp_z_past`, equal duration weights, spread of 12m local-currency excess
returns (FX-hedged by construction). Overlapping series → mean with NW(12); honest Sharpe from
the 12 non-overlapping calendar tracks. Pseudo-OOS (revised MEI inputs).

In [14]:
if not PANEL_OK:
    print("(skipped - requires g10_panel_model_objects.csv)")
else:
    ls = cross_sectional_ls(PANEL, k=LS_TOP_K, min_countries=LS_MIN_COUNTRIES)
    s3 = ls_summary(ls, k=LS_TOP_K)
    print(f"months {s3['months']} | gross ann mean {s3['gross_ann_mean']*100:.2f}% | NW t {s3['NW_t']:.2f}")
    print(f"per-1SD-signal-spread {s3['per_signal_SD']*100:.2f}%  "
          f"(compare paper FMB beta 0.81%)")
    print(f"12-track median SR {s3['median_track_SR']:.2f} | positive tracks {s3['tracks_positive']}")
    print(f"membership changes/month {s3['events_per_month']:.2f} | breakeven {s3['breakeven_bp']:.0f}bp")
    print("net annual mean @ 1/2/5bp: " + " / ".join(
        f"{s3[f'net_ann_mean_c{int(c)}bp']*100:.2f}%" for c in [1, 2, 5]))
    ls2 = cross_sectional_ls(PANEL, k=2, min_countries=LS_MIN_COUNTRIES)
    s2 = ls_summary(ls2, k=2)
    print(f"k=2 sensitivity: gross {s2['gross_ann_mean']*100:.2f}% NW t {s2['NW_t']:.2f}")
    ok = s3["NW_t"] >= 1.645
    print("\nVERDICT (pre-registered):",
          "gross spread significant at 5% one-sided -> add the Section-7 table (gross+net, pseudo-OOS label)."
          if ok else ("gross spread at 10% only -> report as descriptive, one line."
                      if s3["NW_t"] >= 1.282 else
                      "spread not significant -> report the null; the TWFE slope remains the panel claim."))

months 363 | gross ann mean 0.96% | NW t 2.80
per-1SD-signal-spread 0.97%  (compare paper FMB beta 0.81%)
12-track median SR 0.37 | positive tracks 12/12
membership changes/month 0.44 | breakeven 55bp
net annual mean @ 1/2/5bp: 0.94% / 0.93% / 0.87%
k=2 sensitivity: gross 1.42% NW t 3.48

VERDICT (pre-registered): gross spread significant at 5% one-sided -> add the Section-7 table (gross+net, pseudo-OOS label).


## Folding into the paper — exact sentences per outcome

**A (vintage-robust):** replace the §7 *Publication timing* paragraph with: “Rebuilding the anchor
from ALFRED CPI vintages — the full history re-derived at each origin from the vintage then
available — moves the basket $R^2_{OOS}$ from 0.0163 to [x] and leaves the ten-year nested
Clark–West at [y]; the US branch is therefore real-time to data revisions, not only to release
timing.” Then delete “pseudo” from the US rows in the abstract and §4.
**A (degraded):** keep the asterisk; add the measured gap to Limitation 5 with the two numbers.

**B (publishable):** one sentence in §7 *Economic value*: “At a 2bp round-trip cost per unit of
duration notional, the linear rule retains a net Sharpe of [x] versus [y] static, with breakeven
at [z]bp.” **B (not established):** keep gross framing; report breakeven only.

**C (significant):** a five-row table in §7: gross mean, NW *t*, track Sharpe, net at 2bp,
breakeven — captioned as the tradable translation of the Table-7 within-time slope, pseudo-OOS.
**C (null):** one line reporting the null; the within-time inference already carries the claim.